<a href="https://colab.research.google.com/github/bzhuang2-create/SURF---MC-Simulation-for-Educational-Research/blob/main/MC_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <your section title> MC Implementation

This is a MC simulation designed for 06-310 Molecular Foundations of Chemical Engineering. This MC simulation operates according to the Metropolis algorithm, and keeps number (density), volume, and temperature constant.

The main function for this simulation and its helpers can be found by expanding this section and unhiding its individual cells.

By default, the Lennard-Jones pair potential is used for all calculations, though it is possible to specify different pair potential functions.

Below the implmentation section is the section with all of the learning modules. Each module, when ran, will temporarily hide its UI until the simulation finishes running. Note that the species and compositions may vary between modules.

In [1]:
# @title ASE and ipywidgets Install
from google.colab import output
from IPython.display import clear_output
%pip install ase
%pip install -q ipywidgets
%pip install ovito
#!curl -fsSL https://install.julialang.org | sh
#%pip install pyclapeyron
%pip install -c numba icc_rt
%pip install line_profiler
clear_output()

In [2]:
# @title Imports and Constants
%matplotlib inline

import ase
from ase import Atoms, Atom
from ase.calculators.lj import LennardJones as LJ
from ase.geometry.rdf import get_rdf
from ase.io import write
from ase.neighborlist import NeighborList
from ase.visualize.plot import plot_atoms

# note: delete this later to speed up execution time for initial imports
#from pyclapeyron import LJRef
#from pyclapeyron import pressure as reference_pressure

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.widgets import Button, Slider
from bqplot import Figure, Lines, LinearScale

import math
import numpy as np
from scipy.optimize import curve_fit
from scipy.interpolate import make_interp_spline, BSpline
from scipy.stats import expon, boltzmann, rv_continuous, fit
import random

from tqdm import tqdm
import timeit
import pandas as pd
from numba import njit, jit, types, prange
from numba.experimental import jitclass
from numba.typed import Dict

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual, Layout
from ipywidgets import IntSlider, VBox
from IPython.display import clear_output

import io
import base64
from copy import deepcopy
from PIL import Image
from IPython.display import display, HTML

from google.colab import output

kB = 8.617e-5  # eV/K
R = 8.314 # J / mol K

clear_output()

In [3]:
# @title Reference Positions Close to Equilibrium

'''
The number of reference positions, the density at which the equilibrium positions were found, and a conversion of the reference density to atoms / A^3
'''
reference_no_atoms = 256
raw_density = 1202.05 # kg / m^3 (Argon)
reference_density = raw_density * 1000 / 39.948 * 1e-10 * 1e-10 * 1e-10 * 6.022e23 # kg to grams, mols of Argon per gram, meters to A, mols to atoms

'''
A set of reference positions for 256 Argon atoms (sigma = 3.4 and epsilon = 0.01034)
If choosing to initialize with reference positions, these positions will be used. If more than 256 atoms are present, the remaining atoms will be given random positions.
If the system density is different than the reference density, the positions of the atoms will be rescaled accordingly
'''
reference_positions = [[16.136135917648527, 0.2677767806383025, 14.484597179012773], [12.937880377954954, 8.060427641592874, 5.77775425983551], [21.593069899573447, 3.6702773182170207, 0.884621977464709], [21.594677157934004, 9.967561872758512, 0.0607204490144575], [20.285348344674905, 21.943360122759035, 10.044441096637398], [10.993927451501577, 7.910278988309197, 2.789453558724555], [12.872215173088346, 0.028810311539814393, 16.409010401478483], [4.615097044821785, 9.916215881187991, 10.045904091296462], [18.66294882231339, 9.43883875360453, 2.796873430458718], [10.922699741312249, 13.84410563443472, 19.278147774647646], [9.33504221322773, 9.248533558703253, 22.11771332074007], [13.309276279574126, 17.724175360058776, 5.067810891470961], [14.899982439743395, 5.147544457639387, 5.94945065881155], [24.050862198292734, 2.7394926577290777, 8.221339300472524], [11.967050632262005, 12.689183187813454, 12.77366056369077], [20.348503382964843, 17.853983763906506, 3.756111905047268], [22.90264188688966, 18.61473603538731, 14.674879353464041], [22.866739206711753, 17.652294278871988, 22.44749507448135], [10.112228724867347, 11.045848659361399, 4.717591332595694], [12.4088929637115, 11.408933161089783, 1.9965037670267176], [19.519283045344334, 19.085244577988714, 21.51866499054986], [15.94494637791304, 3.8141930911201545, 20.57986328161802], [17.762970653738364, 19.118857947066253, 9.48288141110294], [4.504353448888786, 7.547804210282297, 15.841171660849357], [11.825533499373034, 0.6069576699535166, 22.813587294877042], [22.131895005670305, 0.21352436414521914, 22.980068117413733], [15.65011666588227, 10.837527774004597, 8.944968909579213], [10.26735864105028, 4.236519952011958, 17.82505835541946], [16.737421401325616, 2.09688832890631, 6.808997366943079], [11.01951775561075, 2.757427821528304, 1.7098415984498878], [17.763019115551273, 18.61266310730508, 18.29275802544805], [16.568274354021618, 9.016107577548263, 23.929728711348545], [14.754824392530002, 7.210624064284551, 8.841386656761909], [17.487810198113916, 6.928887677199873, 18.70644605315395], [11.609189569122329, 2.2971595253653487, 12.087040680546174], [17.890630322069523, 1.1447878783054395, 10.621129555233598], [22.017791868705626, 21.278496677717325, 4.030632915394557], [19.55757949444033, 8.490969465997392, 21.550574936739366], [15.9556812705273, 23.239001832197285, 18.44233489755145], [23.689969525914638, 12.635553539960789, 19.112730254875018], [18.403952107179137, 16.886589017105972, 6.688078711328589], [11.153257231486464, 12.525252846793123, 22.895501503518762], [21.842608980929604, 8.395993626801747, 9.829809481445707], [18.734331488185592, 13.320900774444645, 13.873249471093771], [17.301730606000636, 13.769870954603574, 21.138228173871244], [19.574354675169154, 23.52192268773976, 13.30597073208488], [14.595612413659952, 20.392646653704762, 2.8733184902005897], [19.76952170376925, 7.339458407754115, 12.584925430566049], [14.185113704334796, 24.036953546665675, 11.132133164474844], [18.638402098919062, 12.064717046925436, 0.012197828998349536], [16.3505979341792, 5.383897424581764, 23.986184335405188], [19.245028694093516, 3.960405932905361, 17.539400004661726], [15.402669121509767, 13.327704322049978, 11.901238262852175], [11.240888949762896, 6.588074015089963, 20.60530771389106], [22.382764195060492, 1.0332968754804097, 14.775122469311235], [0.8255851765436819, 5.76621043791296, 19.722475346174498], [16.18594137806087, 19.722646730365568, 6.21957842121737], [19.978221811662642, 20.23221909396122, 6.798475181576489], [17.03121574086626, 14.32771493439387, 8.820332995135136], [15.912664342664273, 12.076435741485055, 2.5240343292444036], [21.909100181766238, 1.0065252165721075, 11.04492635325146], [18.259273573850685, 2.017424916071792, 23.675572739698097], [6.610494750955217, 7.083345657735683, 8.84203207347152], [17.410076001462897, 2.241654886025648, 3.1977925765896313], [0.5988579086532916, 18.019538471551915, 11.47007846198355], [19.301602633100263, 21.95123005851115, 19.09003277027174], [1.948156799207775, 5.666717053373021, 7.057708305370934], [4.012140581766488, 4.811557155533379, 10.216422832697916], [12.933626385854403, 3.463374820983689, 15.19773836499164], [14.159538526339754, 20.930933893577304, 9.408636765779045], [14.439647462480982, 12.822342075852589, 19.0785687309982], [10.794946312475032, 9.315165223023325, 13.845877448527727], [21.039328697514524, 6.777703218766845, 3.8289158847884384], [20.556361340786758, 3.3720542174251382, 4.446459336829649], [5.377802722135458, 10.254699079938002, 13.499096522223489], [10.121778706971194, 5.593812302161057, 14.178157776103095], [20.817336374633474, 4.654625551770571, 10.612207782051836], [1.1507340468720204, 2.272193879205226, 11.920470230558458], [21.50443189038895, 11.553002168859763, 3.4684835656574213], [14.597650450227414, 17.219511122238355, 8.425680292256919], [15.478458625146958, 14.762792628463224, 5.3640059532531446], [21.179384344863415, 18.480601897099024, 10.700941306191071], [13.546815823237504, 13.760430254729503, 8.626800386113187], [12.574916244889792, 15.930320730894715, 11.488084528321842], [5.488423869336867, 10.939683009354516, 17.351131897498107], [17.349819795491435, 5.949076171303905, 3.20504526477198], [12.87405777876969, 2.0724775278382985, 19.554225479563573], [11.251370950014735, 4.718169178312288, 4.903390342835558], [9.02422848757087, 10.502229942540652, 18.622541752009838], [11.54063035047082, 10.458906237309938, 8.173520989721936], [19.969298785658673, 16.533048981950028, 15.6134415398701], [10.858626705014945, 22.453749862628403, 9.479277737306276], [0.6555542103716063, 3.174333960941938, 23.0811336015013], [8.954602351735366, 4.70095188432654, 10.710770664628958], [18.781431539642995, 1.3299441373814422, 20.23847550215715], [12.685087875164795, 5.723466693322875, 11.648872262420582], [9.027035423242681, 14.962314303520865, 12.531071935757767], [10.920915840917681, 22.921390329472977, 13.336885070531359], [10.03965521729507, 5.960663375944158, 24.02066727410578], [12.793186803808025, 17.910860257636305, 14.936352408046268], [20.762724846589638, 14.972208964280505, 9.660263827041323], [15.384439569562488, 0.4115673073690604, 21.83599426371567], [13.248982735993, 15.052279351297434, 2.2524365587527426], [18.872674900292587, 14.341698539866412, 3.1005412164569104], [16.485582496933006, 5.01867252307008, 15.356819610634028], [9.770048286869603, 13.980768642468215, 1.906928047177204], [13.011446823011639, 4.301018266447486, 22.723063011813153], [13.927101858964875, 2.225298238780903, 4.253886353465878], [19.626386316248116, 6.891162799334373, 0.49206838354923454], [22.275578544963032, 15.438332921210426, 12.899268135868569], [20.3237875832142, 9.222812783935494, 6.3612662449464095], [1.2113994864407078, 8.791108212301767, 9.024987443229524], [16.01567614991445, 6.980608188087786, 12.152053778933267], [5.183886409806368, 13.682709049577618, 14.582810333759591], [17.73381269793867, 10.373243126681619, 12.106000614295544], [22.699081788796395, 21.763798407585888, 12.704842816815157], [20.22981823757054, 2.3594248855116238, 7.927479264161823], [20.207649257610086, 24.026766889873038, 1.861919184389975], [9.624911373743307, 3.3327127988453227, 21.433949769100167], [14.681615439327942, 2.000307503606597, 0.7313024437533533], [13.211718117546756, 9.554730983639992, 11.240556016929524], [18.688022159469384, 5.554477005463237, 6.6441787398923235], [21.478251382374477, 18.822075309303614, 18.308982269444478], [9.513658976150161, 1.6564446765300833, 15.197730982140317], [11.96531331406276, 11.504418252949714, 16.56135847023016], [23.145160161787324, 11.584415336962316, 8.060186780854377], [20.94132744630189, 6.705161186666031, 15.80767975829951], [12.726818213720366, 22.567123182202973, 19.90257876244793], [11.665564156142672, 19.508093873700226, 11.736725750513195], [23.069722333120403, 9.323106643940136, 20.80464133833356], [23.68076476472041, 20.78751284552413, 9.386874546801348], [2.6608666148470124, 12.29423990705754, 8.102600611169528], [20.388538390314082, 10.200307708218377, 14.712757254170176], [22.176325457227527, 17.38647427669014, 7.232406385823031], [20.386188389531373, 15.670768501675814, 20.092590408192056], [18.356374372804773, 20.986541414079994, 3.364184587792608], [13.633335886109457, 21.549097949597066, 23.45762787650676], [1.5118241153856566, 11.493217886454971, 11.708786956904527], [17.27612313881315, 4.588967777564656, 9.623548752094361], [19.71906087966555, 16.69312359534127, 0.30723222012644785], [17.86757801044297, 12.040708260206923, 5.725192821686686], [1.3228836979032081, 1.862639334417327, 19.47873694590735], [10.476068031672437, 16.644183184408533, 22.834587210030733], [14.594954087011958, 9.055461412068437, 17.730620558990125], [0.910529862346439, 6.489218114276298, 15.99061198851385], [9.938157892475857, 13.828181055337614, 9.094441724131404], [9.895604007797512, 0.33912004105342275, 18.749318324344443], [9.788997102157154, 20.39232215561847, 20.674676026656883], [19.022196581824772, 22.548615656114013, 22.62019283377519], [14.500199785097292, 19.99916124034429, 17.491897959412963], [15.292020297367083, 3.2008459787133487, 12.411370233824254], [17.204951385367163, 11.802541263107063, 16.77317458010052], [14.620847302964021, 14.139056035407592, 15.43051528217051], [19.681349997505595, 9.624108045157744, 18.169253231047264], [14.798608085814513, 7.307482451294445, 21.130468829247526], [18.99087615795817, 0.5429557047461218, 16.70495200546925], [22.582743481943357, 6.363877556162565, 22.800809670967194], [5.016640019685122, 13.455512750065486, 10.993559243630921], [16.70078733066242, 10.280394847677332, 20.270988787699633], [19.424944191108583, 11.613451849406237, 9.00041039238035], [21.46723838077903, 6.457178435364025, 19.257249523078567], [15.159298643725442, 18.474543488345635, 11.903906031032779], [13.503667007842603, 5.369632383421043, 2.053374517079167], [24.139271240899618, 4.4653932642297445, 4.081002222473791], [17.491342605018897, 8.391139379408305, 15.332678302176909], [8.169985351205744, 4.165897149837314, 6.865389894902059], [16.157251330841618, 20.852737587365297, 20.991968898108563], [18.273982761427614, 16.35954427945555, 12.02033607597647], [16.42716364740673, 15.032881196176909, 0.27650314466021275], [14.055936372403613, 11.509818335874588, 5.5980573242297105], [22.09358351556081, 3.123034959082652, 20.637188177026367], [11.09097186779749, 21.000319015006568, 16.94752628761327], [11.491847808525142, 5.802637937186978, 8.323018179270036], [14.082341384488991, 21.350403014406414, 14.066732857601814], [14.041722460105383, 17.872362288996722, 0.26798660846757116], [8.499444627842964, 11.399422514683998, 11.703845755445213], [22.546885374923733, 21.24621995652937, 21.109821563804065], [10.686679936604278, 0.7999687449164765, 5.0582421095871775], [15.720690242449939, 2.3191590723776696, 17.409867393367715], [19.143100523621754, 4.941014790515996, 21.697187483686122], [8.536543653191064, 11.867298946750244, 15.227516442728934], [7.829569588118212, 6.907168017359259, 19.09923297533183], [16.509438542074736, 17.409569747596244, 15.161090174044077], [17.17412751643302, 19.568701014288692, 0.2237692825978432], [16.46466998636685, 17.185330974343696, 21.37484871722604], [12.423769402485236, 21.24951520453374, 5.971338832606054], [12.7854727025324, 23.500050660755004, 2.291858535951894], [9.198738327134048, 7.781119987500608, 6.147524406888529], [6.963504325267009, 4.7586493953392095, 16.102649749104597], [2.3429583697515928, 8.904350698787116, 18.69614879290065], [6.74831574763541, 2.672989268667255, 12.89330972889497], [13.078628792178495, 18.77071007781283, 20.956214058327483], [22.02188467250437, 11.822506225089631, 11.672066752096187], [16.994043436023123, 21.800176639624055, 11.64297249320441], [14.738755797470281, 11.98644947532352, 22.905392574999993], [23.17018160077584, 0.8104727329818096, 4.104637806224305], [16.633021155066874, 17.40564108229899, 3.0790557621458783], [3.711981984610265, 4.499572884925659, 17.78471501823092], [18.25637956813614, 8.097982236983258, 9.243312220011246], [17.248106415955572, 22.821203162537422, 8.055184644027792], [23.12912124773986, 9.382781020831896, 17.132379677335795], [20.95304515639437, 20.187051305237258, 0.5563908191349491], [2.757738974766657, 7.743655616562693, 12.459496198292753], [19.339950605388058, 3.133948353136125, 13.695767280492488], [11.581307426659537, 14.399396226664008, 5.502891831192252], [12.385028753478903, 10.165120489702765, 20.425819410433913], [24.057183704324295, 7.579712408436739, 1.9227686767450427], [11.126388870148547, 18.64020644845925, 8.082294030035849], [1.7955676464055954, 14.819258181205562, 13.473108305308315], [8.128479927729899, 5.457536513534898, 3.0851368131375025], [3.7008931437696915, 4.48851313938082, 13.927487218714722], [13.817768330980648, 5.392692078204499, 18.116702916854358], [9.85027479965427, 8.122727041663774, 10.547023597614514], [2.3508249662892613, 13.732391769290718, 16.942918740991665], [7.954874593756541, 8.287236074567824, 15.611243493005613], [0.18102448427772955, 5.809884804297788, 10.554622057890118], [10.911890647379382, 14.97544326095765, 15.620489250391914], [22.989153601661403, 4.6566613002101205, 13.677036814216109], [16.681666840336625, 8.585021289914215, 6.030233566813848], [1.6259292399921317, 2.070712656641596, 15.619957440623564], [21.38467793553062, 21.814309294271187, 16.037383850421886], [22.4580608524989, 15.041957414522393, 2.8788197451017186], [10.786914826583663, 17.459852468207426, 18.34366014850055], [21.98426914577974, 13.952621035613507, 23.43329402861139], [23.386371949986017, 8.486104636773705, 13.197658004443639], [15.852942927963293, 23.34985155845657, 4.651673650575137], [2.141448785356308, 10.276089535875029, 15.152439050360337], [13.448129592040384, 15.094829332347508, 21.865633209741883], [0.2527823788162359, 14.548372792465768, 10.131967162288982], [6.348253933993865, 6.778870652294448, 12.585041863996018], [23.26114203343017, 15.709269431880363, 16.91199850792185], [22.302628686300032, 5.905787416446139, 7.228862545468771], [7.8615712392790655, 10.62534036972401, 8.336462420989664], [17.418123633040405, 15.164588052177258, 17.928380449947202], [17.471649038570686, 21.31412122801458, 15.715139483063098], [20.930988614083397, 14.048726802205913, 6.171823592518891], [12.778360019228755, 8.404241132618793, 23.868352118317176], [23.46073971100309, 12.443344007845765, 14.921618727108152], [22.214374441042487, 0.20926867998861182, 18.633457732129976], [23.89416814630228, 8.865103358797127, 5.559519570619557], [22.28378367738283, 23.539533823123925, 7.466912345662545], [20.446686540184412, 12.02831246246263, 20.80841175659828], [19.551011665330126, 19.48577573424558, 13.49623873466641], [14.838718373656858, 8.652043192127383, 2.884494738863208], [19.396611647686633, 24.014779255741544, 5.309205745245436], [20.55165368221832, 13.227231188630723, 17.16605685772462], [8.938784751410601, 10.479902569332388, 1.1882640809047142], [13.640655944812917, 0.042153210902284854, 7.453272626616196], [14.152373536766936, 16.368754222213813, 18.364107959393248], [14.01015908235191, 3.292636839768179, 8.991261083714617], [22.974191672661366, 3.7144153175909853, 17.2362917588285], [11.122872102203928, 7.774293807847506, 17.31133396317891], [10.753013096562189, 2.0165775821427783, 8.642085887426155], [13.598432779162026, 7.016541571650384, 14.775999779430425], [16.437822543908833, 23.12066642815878, 1.0094914081059545], [14.731926138578313, 10.486649472295817, 14.337350453771268]]

In [34]:
# @title Hardcoded Parameters for LJ and Bonding

MAX_ATOMIC_NUMBER = 120
# an important macro for many functions. Can be increased if more chemical groups need to be encoded

'''
The cantor pairing function, which maps two positive integers to a unique integer. Useful for quick data lookup for various species

Note: order matters, but subsequent definition of the property arrays give the correct properties regardless of order
a: first integer
b: second integer

returns: a unique integer
'''
@njit(fastmath=True)
def cantor_pair(a, b):
  return int(((a + b) * (a + b + 1)) // 2 + b)

'''
This is a dictionary setup using numba. Not used directly in the Monte Carlo simulation, but useful to set up the cantor pairing function array
'''
# Lennard-Jones parameter dictionarys
LJ_dict = Dict.empty(key_type = types.unicode_type, value_type = types.UniTuple(types.float64, 3))

# Order: atomic number, sigma, epsilon
LJ_dict['Ar'] = (18, 3.4, 0.01034)
LJ_dict['Cl'] = (17, 3.4, 0.01034)
LJ_dict['Au'] = (79, 3.4, 0.01034)
LJ_dict['Ag'] = (47, 3.4, 0.01034)
LJ_dict['H'] = (1, 1.4, 0.01034)
LJ_dict['C'] = (6, 1.4, 0.01034)
LJ_dict['He'] = (2, 3.4, 0.1034)
LJ_dict['B'] = (5, 3.4, 0.01034)
LJ_dict['N'] = (7, 2.2, 0.01034)
LJ_dict['O'] = (8, 2.4, 0.01034)
LJ_dict['S'] = (16, 3.1, 0.01034)
LJ_dict['Na'] = (11, 2.8, 0.01034)

LJ_dict['U'] = (92, 3.4, 0.01034) # for polymers - do not touch yet
LJ_dict['Al'] = (13, 3.4, 0.01034) # polymers

# list of tuples of the form (atom#1, atom#2, k, ideal bond length)
# pairs don't have to be added both ways because the subsequent code will add both directions
covalent_bond_dict = []
covalent_bond_dict.append((92, 92, 114 * 0.01034, 1.12 * 3.4))
covalent_bond_dict.append((13, 13, 114 * 0.01034, 1.12 * 3.4))

'''
This is the cantor pairing function array. It stores the average LJ parameters for each pair of species as sigma and (4)epsilon to reduce computation later.
If more species are needed, the length of the array can simply be extended
'''
LJ_avg_array = np.zeros((cantor_pair(MAX_ATOMIC_NUMBER, MAX_ATOMIC_NUMBER), 2), dtype = np.float64)  # (sigma, (4)epsilon)
harmonic_bond_array = np.zeros((cantor_pair(MAX_ATOMIC_NUMBER, MAX_ATOMIC_NUMBER), 2), dtype = np.float64) # (k, r_ideal)

'''
Adding all LJ and covalent bond data into the arrays
'''
for key1 in LJ_dict.keys():
  for key2 in LJ_dict.keys():
    number1, sigma1, epsilon1 = LJ_dict[key1]
    number2, sigma2, epsilon2 = LJ_dict[key2]
    LJ_avg_array[cantor_pair(number1, number2)] = [((sigma1 + sigma2) / 2), 4 * np.sqrt(epsilon1 * epsilon2)]

for entry in covalent_bond_dict:
  atom1, atom2, k, r_ideal = entry
  harmonic_bond_array[cantor_pair(atom1, atom2)] = [k, r_ideal]
  harmonic_bond_array[cantor_pair(atom2, atom1)] = [k, r_ideal]

In [5]:
# @title Molecule Trie Definition
# both polymers and molecules will be implemented as the same class
class molecule_trie:
    def init(self, at_number, at_symbol, prev_trie, polymer_end_unit, cyclic_end):
        self.atomic_number = at_number
        # holds the atomic number of the current atom as an int

        self.atomic_symbol = at_symbol
        # holds the atomic symbol of the current atiom as a string

        self.children = []
        # a list of molecule_tries
        # if this list is empty, then this atom has no more next neighbors
        # if this list is not empty, then all next neighbors are in this list

        self.index = -1
        # this number determines which data in arrays such as the position list and atomic number list correspond to this atom
        # start out at -1 because we build molecule tries before we build the other arrays

        self.prev = prev_trie
        # this trie object determines the parent node of the current trie (pointer)
        # None indicates that this is the beginning of the molecule
        # Note: for cyclic molecules, there is no beginning/end

        self.poly_end_unit = polymer_end_unit
        # True: start the next polymer unit from this atom
        # False: do nothing (the default state for most atoms)
        # Note: this isn't used when initializing a MC environment. Instead, it is used only when constructing the polymer

        self.traversal_state = False
        # Lets traversal algorithms know whether or not an atom has been encountered previously
        # Avoids double-counting during traversal
        # The actual value of this boolean doesn't matter because each traversal should flip each atom's boolean exactly once
        # As long as traversal algorithms know the initial state of all the booleans, it can successfully terminate - then all booleans are the opposite of the initial state

In [6]:
# @title Lennard Jones Potential and Harmonic Bonds

'''
A function to calculate the Lennard-Jones interaction energy between two atoms

r: the distance between the two atoms (A)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
lj_avg_array: a long array of average LJ parameters. Indexed using a pairing function, which is faster than accessing a dictionary

returns: the interaction energy between the two atoms (eV)
'''
@njit(fastmath = True)
def lennard_jones(r, atom1, atom2, lj_avg_array):
  sigma, epsilon_4 = lj_avg_array[cantor_pair(atom1, atom2)]
  #sigma and (4)epsilon
  sigma_over_r = sigma / r

  return epsilon_4 * ((sigma_over_r ** 12) - (sigma_over_r ** 6))

'''
The analytical derivative of the Lennard-Jones potential

r: the distance between the two atoms (A)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
lj_avg_array: a long array of average LJ parameters. Indexed using a pairing function, which is faster than accessing a dictionary

returns: the derivative of the interaction energy between the two atoms (eV/A)
'''
@njit(fastmath = True)
def lennard_jones_derivative(r, atom1, atom2, lj_avg_array):
  sigma, epsilon_4 = lj_avg_array[cantor_pair(atom1, atom2)]
  #sigma and (4)epsilon

  return 6 * (epsilon_4 / r) * ((2 * (sigma / r) ** 12) - ((sigma / r) ** 6))
  # negative? sklog wiki didn't have negative

'''
The tail correction factor according to Frenkel and Smit, Equation 3.2.8

density1: the density of the first species
density2: the density of the second species
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
lj_avg_array: a long array of average LJ parameters. Indexed using a pairing function, which is faster than accessing a dictionary
r_cutoff: the cutoff distance (A)

returns: the tail correction factor (eV / A^3)
'''
@njit(fastmath = True)
def frenkel_smit_tail_correction(density1, density2, atom1, atom2, lj_avg_array, r_cutoff):
  sigma, epsilon_4 = lj_avg_array[cantor_pair(atom1, atom2)]
  #sigma and (4)epsilon
  return 4 / 3 * np.pi * density1 * density2 * epsilon_4 * (sigma ** 3) * ((2 / 3 * ((sigma / r_cutoff) ** 9)) - ((sigma / r_cutoff) ** 3))

'''
Calculates potential energy according to the harmonic bond equation

r: the distance between the two atoms (A)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
bond_param_array: a long array of harmonic bond parameters. Indexed using the cantor pairing function
'''
@njit(fastmath = True)
def harmonic_bond(r, atom1, atom2, bond_param_array):
  k, r_ideal = bond_param_array[cantor_pair(atom1, atom2)]
  return 0.5 * k * ((r - r_ideal) ** 2)

@njit(fastmath = True)
def harmonic_bond_derivative(r, atom1, atom2, bond_param_array):
  k, r_ideal = bond_param_array[cantor_pair(atom1, atom2)]
  return k * (r - r_ideal)

In [7]:
# @title Single and Total Potential Energy Calculations
'''
Calculates all interaction energies for a single atom, excluding itself

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
no_atoms: total number of atoms
box_length: length of the cube (A)
selected: the index of the atom to calculate energies for
backward_pos, forward_pos: for molecules, indicates when we should stop calculating intermolecular energy potential
direct_children: a 1D array of children
direct_parent: the parent, if any (-1 indicates no parent)

energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

returns: the total interaction energies for a single atom
'''
@njit(fastmath = True)
def single_energy(atom_numbers, atom_positions, direct_children_array, direct_parent_array,
                  no_atoms, box_length, selected, backward_pos, forward_pos,
                  energy_fxn, energy_params, bond_energy_fxn, bond_energy_params):
  # Calculates the interaction energies for a selected atom
  r_cutoff = box_length / 2.01 # what to put for this?
  fixed_position = atom_positions[selected]
  direct_children = direct_children_array[selected]
  direct_parent = direct_parent_array[selected]

  energy = 0
  for i in range(backward_pos):
    if(i == selected):
      continue

    other_position = atom_positions[i]

    dx = fixed_position[0] - other_position[0]
    dy = fixed_position[1] - other_position[1]
    dz = fixed_position[2] - other_position[2]

    # enforce periodic boundary conditions
    dx -= box_length * round(dx / box_length)
    dy -= box_length * round(dy / box_length)
    dz -= box_length * round(dz / box_length)
    r = np.sqrt(dx * dx + dy * dy + dz * dz)

    if(r > r_cutoff):
      continue
    energy += energy_fxn(r, atom_numbers[selected], atom_numbers[i], energy_params)

  for i in range(forward_pos, no_atoms):
    if(i == selected):
      continue
    other_position = atom_positions[i]

    dx = fixed_position[0] - other_position[0]
    dy = fixed_position[1] - other_position[1]
    dz = fixed_position[2] - other_position[2]

    # enforce periodic boundary conditions
    dx -= box_length * round(dx / box_length)
    dy -= box_length * round(dy / box_length)
    dz -= box_length * round(dz / box_length)
    r = np.sqrt(dx * dx + dy * dy + dz * dz)

    if(r > r_cutoff):
      continue
    energy += energy_fxn(r, atom_numbers[selected], atom_numbers[i], energy_params)

  for child in direct_children:
    if(child < 0):
      break
    child_position = atom_positions[child]

    dx = fixed_position[0] - child_position[0]
    dy = fixed_position[1] - child_position[1]
    dz = fixed_position[2] - child_position[2]

    # enforce periodic boundary conditions
    dx -= box_length * round(dx / box_length)
    dy -= box_length * round(dy / box_length)
    dz -= box_length * round(dz / box_length)
    r = np.sqrt(dx * dx + dy * dy + dz * dz)

    energy += bond_energy_fxn(r, atom_numbers[selected], atom_numbers[child], bond_energy_params)

  if(direct_parent != -1):
    parent_position = atom_positions[direct_parent]

    dx = fixed_position[0] - parent_position[0]
    dy = fixed_position[1] - parent_position[1]
    dz = fixed_position[2] - parent_position[2]

    # enforce periodic boundary conditions
    dx -= box_length * round(dx / box_length)
    dy -= box_length * round(dy / box_length)
    dz -= box_length * round(dz / box_length)
    r = np.sqrt(dx * dx + dy * dy + dz * dz)

    energy += bond_energy_fxn(r, atom_numbers[selected], atom_numbers[direct_parent], bond_energy_params)

  return energy

'''
Calculates the total energy of the system by finding all pairwise interaction energies and dividing by 2 to avoid double-counting

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
no_atoms: total number of atoms
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

returns: the total interaction energies for a system
'''
@njit(fastmath = True)
def total_energy(atom_numbers, atom_positions, direct_children_array, direct_parent_array, molecule_index_array,
                 no_atoms, box_length,
                 energy_fxn, energy_params, bond_energy_fxn, bond_energy_params):
  energy = 0
  for i in range(no_atoms):
    backward_pos, forward_pos = molecule_index_array[i]
    if(backward_pos == -1):
      backward_pos = i
      forward_pos = i

    energy += single_energy(atom_numbers, atom_positions, direct_children_array, direct_parent_array,
                            no_atoms, box_length, i, backward_pos, forward_pos,
                            energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)

  return energy / 2

In [8]:
# @title Pressure Calculations and Partial RDFs

'''
Converts a distance to a bin index for an RDF

distance: the distance to convert
R_bins: a list of distances corresponding to the RDF

returns: the index of the bin the distance is in
'''
@njit(fastmath = True)
def distance_to_bin(distance, R_bins):
  single_radius = R_bins[1] - R_bins[0]
  # this assumes there are at least two bins, which there should be if the slider minimums are appropriate

  index = np.floor(distance / single_radius)
  return int(index)

'''
Generates a partial RDF for a specific pair of species in the system. Does not check both directions

selected_species: the atomic symbol for the first atom
target_species: the atomic symbol for the second atom
species_list: list of names of atoms (ex. ['H', 'H', 'C'])
no_atoms: total number of atoms
atom_positions: list of atomic positions, corresponding to species_list
box_length: length of the cube (A)
R_bins: a list of distances corresponding to the RDF

returns: a single partial RDF for the two species
'''
@njit(fastmath = True)
def partial_RDF(selected_species, target_species, species_list, no_atoms, atom_positions, box_length, R_bins):
  histogram = np.zeros(len(R_bins))
  a_select_count = 0
  b_select_count = 0
  volume = box_length ** 3

  for i in range(no_atoms):
    if(species_list[i] == selected_species):
      selected_position = atom_positions[i]
      a_select_count += 1
      b_select_count = 0

      for j in range(no_atoms):
        if(species_list[j] == target_species):
          target_position = atom_positions[j]
          b_select_count += 1

          dx = selected_position[0] - target_position[0]
          dy = selected_position[1] - target_position[1]
          dz = selected_position[2] - target_position[2]

          dx -= box_length * round(dx / box_length)
          dy -= box_length * round(dy / box_length)
          dz -= box_length * round(dz / box_length)

          distance = np.sqrt(dx*dx + dy*dy + dz*dz)
          index = distance_to_bin(distance, R_bins)

          if(index < len(histogram)):
            histogram[index] += 1

  histogram = histogram / max(a_select_count * b_select_count, 1) * volume
  for i in range(len(histogram)):
    radius = R_bins[i]
    if(radius == 0):
      histogram[i] = 0
    else:
      histogram[i] = histogram[i] / (4 * np.pi * (radius ** 2) * (R_bins[1] - R_bins[0]))

  return histogram

'''
Generates a list of bucket distances for an RDF given the length of the box and the number of bins

box_length: length of the cube (A)
no_bins: number of bins

returns: a list of distances corresponding for an RDF
'''
@njit(fastmath = True)
def bins_to_distance(box_length, no_bins):
  distances = np.zeros(no_bins)
  for i in range(no_bins):
    distances[i] = i * (box_length / 2.01) / no_bins
  return distances

'''
This is the radial distribution function that is assumed at very low densities
g(r) = exp(-E(r) / kB * T)

radius: the distance between the two atoms
temperature: the temperature of the system (K)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
params: any additional parameters the user-specified energy function needs
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)

returns: the value of the radial distribution function
'''
@njit(fastmath = True)
def mayer_f_fxn(radius, temperature, atom1, atom2, energy_params, energy_fxn):
  if (radius <= 0):
    radius = 1e-6
  return np.exp(-energy_fxn(radius, atom1, atom2, energy_params) / (kB * temperature))

'''
Calculates the various integrands of the pressure equation. Does NOT include weights by molar fraction or the 2/3(pi) coefficient in front

atom1: atomic number of the first atom type
atom2: atomic number of the second atom type
RDF: a list corresponding to the RDF for the specific atom1/atom2 combination
R_bins: a list of distances corresponding to the RDF
params: any additional parameters the user-specified energy function needs
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_fxn_derivative: the derivative of energy_fxn w/r/t r. Must be of the form fxn(radius, atom#1, atom#2, params)
mayer: whether to use the mayer f-fxn simplification for calculating pressure

returns: the value of the integrand
'''
@njit(fastmath = True)
def pressure_equation(temperature, atom1, atom2, RDF, R_bins, energy_params, energy_fxn = lennard_jones, energy_fxn_derivative = lennard_jones_derivative, mayer = False):
  sum = 0
  for i in range(len(R_bins) - 1):
    r1 = R_bins[i]
    r2 = R_bins[i + 1]
    if(r1 == 0 or r2 == 0):
      continue

    dEdr2 = energy_fxn_derivative(r2, atom1, atom2, energy_params)
    dEdr1 = energy_fxn_derivative(r1, atom1, atom2, energy_params)

    if(mayer):
      gr2 = mayer_f_fxn(r2, temperature, atom1, atom2, energy_params, energy_fxn)
      gr1 = mayer_f_fxn(r1, temperature, atom1, atom2, energy_params, energy_fxn)
    else:
      gr2 = RDF[i + 1]
      gr1 = RDF[i]

    total2 = (r2 ** 3) * gr2 * dEdr2
    total1 = (r1 ** 3) * gr1 * dEdr1
    sum += (total1 + total2) * (r2 - r1) / 2

  return sum


'''
Because molecules contain multiple atoms and the average RDF needs to be weighted by counts of atoms, this function exists to convert the atomic_number

atomic_number_list: list of atomic numbers

returns: an array where the index == atomic number and the entry == fraction (in terms of total atoms in the system)
'''
def generate_atomic_fractions(atomic_number_list):
  unique_numbers = (pd.unique(pd.Series(atomic_number_list)))
  total_atoms = len(atomic_number_list)
  output_array = np.zeros(MAX_ATOMIC_NUMBER)

  for number in unique_numbers:
    count = np.count_nonzero(atomic_number_list == number)
    output_array[number] = (count / total_atoms)
  return output_array


'''
Note: by default, the cutoff radius is box_length / 2.01. If this is changed anywhere, need to update it in the partial RDF generation as well

tail_correction_fxn: a function to calculate the tail correction factor. Must be of the form fxn(density1, density2, atom#1, atom#2, params, cutoff_radius)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
fraction1: the mol fraction of the first atom
fraction2: the mol fraction of the second atom
params: any additional parameters the user-specified energy function needs
cutoff_radius: the cutoff radius (A)
density: the density of the system (atoms per A^3)

returns: the tail correction factor
'''
def tail_correction(tail_correction_fxn, atom1, atom2, fraction1, fraction2, energy_params, cutoff_radius, density):
  return tail_correction_fxn(fraction1 * density, fraction2 * density, atom1, atom2, energy_params, cutoff_radius)


'''
Given a distance and an energy function, calculates the pairwise force between two atoms

r: the distance between the two atoms (A)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
params: any additional parameters the user-specified energy function needs

returns: the pairwise force between the two atoms (eV/A)
'''
@njit(fastmath = True)
def pairwise_force(r, atom1, atom2, energy_params):
  dEdr = lennard_jones_derivative(r, atom1, atom2, energy_params)
  return dEdr
# NOTE: if time allows, make this more generic

'''
calculates the 1/(3V)... term in the virial pressure equation

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

returns: the 1/(3V) term in the virial pressure equation (eV)
'''
@njit(fastmath = True)
def virial_pressure(atom_numbers, atom_positions, box_length, energy_fxn, energy_params):
  sum = 0

  for i in range(len(atom_numbers)):
    for j in range(len(atom_numbers)):
      if(i != j):

        atom1 = atom_numbers[i]
        atom2 = atom_numbers[j]

        dx = atom_positions[i][0] - atom_positions[j][0]
        dy = atom_positions[i][1] - atom_positions[j][1]
        dz = atom_positions[i][2] - atom_positions[j][2]

        dx -= box_length * round(dx / box_length)
        dy -= box_length * round(dy / box_length)
        dz -= box_length * round(dz / box_length)

        r = np.sqrt(dx * dx + dy * dy + dz * dz)
        force = pairwise_force(r, atom1, atom2, energy_params)
        sum += r * force

  return sum / (6 * box_length ** 3)
  #dividing by 6 avoids double counting

In [9]:
# @title Molecule Trie Traversal Helpers

'''
Creates a unit vector pointing in a random direction
returns: a numpy array serving as the unit vector
'''
@njit(fastmath=True)
def unit():
  vector = np.array([np.random.uniform(-1, 1), np.random.uniform(-1, 1), np.random.uniform(-1, 1)])
  return vector / np.linalg.norm(vector)


'''
Placeholder function
'''
@njit(fastmath = True)
def check_lone_pairs(children_list):
  return 0

'''
Given a list of children generate vectors for the molecule's children
Current implmentation is for polymers only, so there are no lone pairs (change when molecules are implemented)

self_identity: the atomic number of the parent atom
children_identity: the atomic numbers of the children atoms (list of ints)
prev_vector: a unit vector pointing FROM the previous atom TO this atom (self).
  If this is None, then this is the first atom of the molecule and the bond(s) can be oriented in a random direction

returns: a list of vectors (not unit vectors) pointing FROM this atom TO its children
  the list of vectors is in the same order as the children_identity list
'''
def VSEPR(self_identity, children_identity, prev_normal_vector = None):
  no_children = len(children_identity)
  lone_pairs = 0
  no_bonds = no_children - lone_pairs

  if(prev_normal_vector is None):
    prev_normal_vector = unit()

  match (no_bonds, lone_pairs):
    case (0, 0):
      # End of branch/molecule
      return []

    case (1, 0):
      # Linear Geometry (the only one that's implemented as of now)
      # consider changing this, though initilization shouldn't matter if convergence happens quickly
      bond_k, ideal_bond_length = harmonic_bond_array[cantor_pair(self_identity, children_identity[0])]
      return [prev_normal_vector * ideal_bond_length]

    case (_, _):
      assert(False)


'''
Given a molecule/polymer trie, traverse it and add the necessary elements to the positions array and species string
Keeps track of how many atoms were added (keeps track of which species in the species string are part of the same molecule/polymer)
Placement procedure (without Flory Huggins):
  the first atom of the trie is place in a random location
  the remaining atoms are placed according to VSEPR (note: this means that for molecules lone pairs will have to be modeled)
    additional note: do not make straight chain polymers longer than the box! (until folding/bending is implemented)

self_trie: the trie to traverse (should be a deep copy of the one provided by the user)
positions: list of atomic positions
species_string: string of atomic symbols
current_index: the index in positions and species_string where data will be placed. Faster than calling len(positions)
traversal_state: tracks which value the trie(s)' traversal state had initially (see molecule_trie documentation)
box_length: length of the cube (A). Useful for placement of atoms and enforcement of periodic boundary conditions
prev_position: the xyz coordinates of the previous atom
prev_vector: a vector that points FROM prev_position TO the current atom

returns:
  self_trie: the same trie that was provided, except relevant data has been added to its fields
  positions: list of atomic positions
  species_string: string of atomic symbols
  atoms_added: the number of atoms that were added (including recursive calls)
  direct_children_list: a list of indices that correspond to the atoms which are direct children of this atom. It will be padded with -1 to ensure proper conversion to a numpy array
    Because all indices are unique, it is always safe and correct to call np.unique on them
'''
def traverse_trie_placement(self_trie, positions, species_string, direct_children_list, given_index, traversal_state, box_length, prev_position = None, prev_vector = None):
  atom_add_count = 0
  direct_children_list.append([])
  # always append at the beginning of this function so that the list is the correct length for recursive calls

  if(self_trie.traversal_state != traversal_state):
  # Case 1: this trie has already been encountered
    assert self_trie.index != -1, f"self_trie.index: {self_trie.index} -1"
    return (self_trie, atom_add_count, positions, species_string, [self_trie.index])

  symbol = self_trie.atomic_symbol
  species_string += symbol

  self_trie.index = given_index # this index of THIS atom
  atom_add_count += 1
  self_trie.traversal_state = not traversal_state

  self_identity = self_trie.atomic_number
  children_identity = []
  for child in self_trie.children:
    children_identity.append(child.atomic_number)


  if(self_trie.prev is None):
    assert prev_position is None and prev_vector is None, f"prev_position: {prev_position} prev_vector: {prev_vector}"
    # Case 2: this trie represents the first atom of the molecule
    self_position = [np.random.rand() * box_length, np.random.rand() * box_length, np.random.rand() * box_length]
    positions.append(self_position)

    # place children according to VSEPR, if any
    bond_vectors = VSEPR(self_identity, children_identity)

  else:
    assert prev_vector is not None and prev_position is not None, f"prev_position: {prev_position} prev_vector: {prev_vector}"
    # Case 3: this trie has a previous/parent atom
    self_position = prev_position + prev_vector
    self_position[0] %= box_length
    self_position[1] %= box_length
    self_position[2] %= box_length
    positions.append(self_position)

    # place children according to VSEPR, taking into account that this molecule already has one bond
    prev_normal_vector = prev_vector / np.linalg.norm(prev_vector)
    bond_vectors = VSEPR(self_identity, children_identity, prev_normal_vector = prev_normal_vector)

  child_index = given_index + 1 # the index of the first child of THIS atom

  for i in range(len(children_identity)):
    child = self_trie.children[i]
    (child_trie, child_add_count, positions, species_string, direct_children_list) = traverse_trie_placement(child, positions, species_string, direct_children_list, child_index, traversal_state, box_length,
                                                                                       prev_position = self_position, prev_vector = bond_vectors[i])
    atom_add_count += child_add_count
    child_index += child_add_count # because indices are added sequentially, the number of children added all occupy that amount of next available indices
    direct_children_list[given_index].append(child_trie.index)

  return (self_trie, atom_add_count, positions, species_string, direct_children_list)


'''
returns: an array of the form [-1, -1, 4, 7] ==> atoms 0 and 1 have no parents. Atom 4 is the parent of atom 2. Atom 7 is the parent of atom 3
'''
def generate_direct_parent_list(direct_children_list):
  direct_parent_list = np.full(len(direct_children_list), -1)
  for i in range(len(direct_children_list)):
    child_list = direct_children_list[i]
    for child in child_list:
      if(child == -1):
        break
      direct_parent_list[child] = i
  return direct_parent_list


In [52]:
# @title Simulation Initialization
'''
Converts a given number of atoms and number density to the corresponding cube length

no_atoms: total number of atoms
rho: density (atoms per A^3)
returns: the length of the cube (A)
'''
@njit(fastmath = True)
def density_to_length(no_atoms, rho):
  Volume = no_atoms / rho
  Length = np.cbrt(Volume)
  return Length

def nested_list_to_numpy(given_list):
  max_len = max(len(row) for row in given_list)
  padded = [row + [-1] * (max_len - len(row)) for row in given_list]
  numpy_array = np.array(padded, dtype = np.int32)
  return numpy_array

'''
Initializes an ASE Atoms object for the Monte Carlo Simulation

no_objects: total number of atoms and/or molecules
length: length of the cube
species: list of tuples of the type (Atomic Symbol(s), Mol Fraction, Object Type)
  single atoms: (Atomic Symbol, Mol Fraction, 'Atom') (string, float, string)
  molecules: (Atomic Symbol Trie, Mol Fraction, 'Molecule') (molecule_trie, float, string)

reference_start: whether to use the equilibrium reference positions

returns: an ASE Atoms object, with all atoms placed in random locations inside the box
'''
def initialize_objects(no_objects, length, species, reference_start):

  total_atoms_added = 0
  total_objects_added = 0
  # atoms tracks how many atoms are added (ex. UU would be two)
  # objects tracks how many objects are added (ex. UU would be one). A single atom (ex. Argon) counts as a single atom and one object

  positions = [] # list of vectors representing positions for each atom
  species_string = "" # string of all atomic symbols. Which ones are part of the same molecule can be found in molecule_index_array
  molecule_index_array = [] # [(-1, -1), (-1, -1), (2, 4), (2, 4), (2, 4)] ==> the first two atoms are singleton atoms. Atoms 3-5 are part of the same molecule
  molecule_object_array = []
  # holds either None or molecule_trie objects. Useful for determining the connectivity of atoms.
  # the index of the entry corresponds to its unique_molecule_count index in the molecule_index_array
  # None: this entry is a singleton atom
  # molecule_trie: deep copy of the prototype trie provided in the species tuples

  direct_children_list = []
  # NOT: this currently means that the maximum number of bonds/lone pairs an atom can have is 8 (unlikely to be exceeded in the context of this course)

  for element in species:
    symbol, fraction, object_type = element
    object_add_count = int(fraction * no_objects)

    # right now only single atoms are permitted to start from reference positions
    # initializing polymers/molecules from reference positions may not have the intended effect
    match object_type:
      case 'Atom':
        if(reference_start):
          reference_length = density_to_length(reference_no_atoms, reference_density)
          scaling_factor = length / reference_length

          for i in range(min(object_add_count, reference_no_atoms)):
            positions.append([reference_positions[i][0] * scaling_factor, reference_positions[i][1] * scaling_factor, reference_positions[i][2] * scaling_factor])
            species_string += symbol
            total_atoms_added += 1
            total_objects_added += 1
            molecule_index_array.append((-1, -1))
            molecule_object_array.append(None)
            direct_children_list.append([])

          for i in range(object_add_count - reference_no_atoms):
            positions.append([np.random.rand() * length, np.random.rand() * length, np.random.rand() * length])
            species_string += symbol
            total_atoms_added += 1
            total_objects_added += 1
            molecule_index_array.append((-1, -1))
            molecule_object_array.append(None)
            direct_children_list.append([])

        else:
          for i in range(object_add_count):
            positions.append([np.random.rand() * length, np.random.rand() * length, np.random.rand() * length])
            species_string += symbol
            total_atoms_added += 1
            total_objects_added += 1
            molecule_index_array.append((-1, -1))
            molecule_object_array.append(None)
            direct_children_list.append([])

      case 'Molecule':
        assert(isinstance(symbol, molecule_trie), f"symbol: {symbol} type: {type(symbol)}")

        for i in range(object_add_count):
          trie_copy = deepcopy(symbol)
          traversal_state = trie_copy.traversal_state
          starting_molecule_index = total_atoms_added

          (trie_copy_updated, atom_add_count, positions, species_string, direct_children_list) = traverse_trie_placement(trie_copy, positions, species_string, direct_children_list,
                                                                                                                         total_atoms_added, traversal_state, length)
          total_atoms_added += atom_add_count
          total_objects_added += 1
          for i in range(atom_add_count):
            molecule_index_array.append((starting_molecule_index, total_atoms_added))
            molecule_object_array.append(trie_copy_updated)
            # many pointers to the same object

  direct_parent_list = (generate_direct_parent_list(direct_children_list))
  direct_children_list = nested_list_to_numpy(direct_children_list)

  # convert molecule_index_array to numpy array
  molecule_index_array = np.array(molecule_index_array, dtype = np.int32)

  MC = Atoms(species_string, positions, cell = [length, length, length], pbc = [True, True, True])
  return (MC, molecule_index_array, molecule_object_array, direct_children_list, direct_parent_list)


'''
Initializes an ASE Atoms object for the Monte Carlo Simulation with all atoms starting on a cubic lattice structure (or as close as possible)
WARNING: only works for monoatomic atoms

no_atoms: total number of atoms
length: length of the cube
species: list of tuples of the type (Atomic Symbol, Mol Fraction, Object Type)

returns: an ASE Atoms object, with all atoms placed in a cubic lattice
'''
def initialize_atoms_cube(no_atoms, length, species):

  dimen = int(np.ceil(no_atoms ** (1/3)))
  step = length / dimen
  positions = []

  atom_add_count_ceiling = 0

  species_string = ""
  for i in range(len(species)):
    assert(species[i][2] == 'Atom')
    atom_add_count = int(species[i][1] * no_atoms)
    atom_add_count_ceiling += atom_add_count
    species_string += species[i][0] * atom_add_count

  for i in range(dimen):
    for j in range(dimen):
      for k in range(dimen):
        index = i * dimen ** 2 + j * dimen + k
        if(index < no_atoms and index < atom_add_count_ceiling):
          positions.append([i * step, j * step, k * step])


  MC = Atoms(species_string, positions, cell = [length, length, length], pbc = [True, True, True])

  molecule_index_array = np.full((atom_add_count_ceiling, 2), -1, dtype = np.int32)
  molecule_object_array = [None] * atom_add_count_ceiling
  direct_children_array = np.full((atom_add_count_ceiling, 1), -1, dtype = np.int32)
  direct_parent_array = np.full(atom_add_count_ceiling, -1, dtype = np.int32)

  return (MC, molecule_index_array, molecule_object_array, direct_children_array, direct_parent_array)

'''
Initializes the monte Carlo simulation

no_atoms: total number of atoms
density: number density of the atoms (atoms per A^3)
species: list of tuples of the type (Atomic Symbol, Mol Fraction, Object Type)
reference_start: whether to use the equilibrium reference positions

returns: an ASE Atoms object, with all atoms placed in random locations inside the box
'''
def MC_initial(no_atoms, density, species, cube_start, reference_start):
  length = density_to_length(no_atoms, density)

  # this means that cube_start will override reference_start
  if(cube_start):
    initial_tuple = initialize_atoms_cube(no_atoms, length, species)
  else:
    initial_tuple = initialize_objects(no_atoms, length, species, reference_start)
  return initial_tuple


<>:94: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:94: SyntaxWarning: assertion is always true, perhaps remove parentheses?
/tmp/ipykernel_17986/2757787527.py:94: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(isinstance(symbol, molecule_trie), f"symbol: {symbol} type: {type(symbol)}")


In [11]:
# @title Energy-Related Helpers

'''
Converts a potential energy value into a bin for the histogram

energy: the potential energy to convert (eV)
bin_size: the size of the bins to use for the histogram (eV)
cutoff_bin: this bin corresponds to zero energy (the bin before this index is the lowest negative energy bin, and this bin in the lowest positive energy bin)

returns: the index of the bin to place the energy in
'''
@njit(fastmath = True)
def energy_to_bin(energy, bin_size, cutoff_bin):

  index_raw = int(energy / bin_size)
  index_adjust = index_raw + cutoff_bin
  return index_adjust

'''
Finds the energy distribution of a system, accounting for all atoms

bin_size: the size of the bins to use for the histogram
no_bins: the number of bins to use for the histogram
cutoff_bins: this bin corresponds to zero energy (the bin before this index is the lowest negative energy bin, and this bin in the lowest positive energy bin)
no_atoms: total number of atoms
atom_positions: list of atom positions
atom_numbers: list of atomic numbers
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

returns: the normalized histogram, bin size, number of bins, and cutoff bin
'''
def find_energy_distribution(bin_size, no_bins, cutoff_bin, no_atoms, atom_positions, atom_numbers, box_length, energy_fxn, energy_params):
  histogram = np.zeros(no_bins)

  for i in range(no_atoms):
    energy = single_energy(atom_numbers, atom_positions, no_atoms, box_length, i, energy_fxn, energy_params)

    index = energy_to_bin(energy, bin_size, cutoff_bin)
    if(index < no_bins):
      histogram[index] += 1
    elif(index >= no_bins):
      histogram[-1] += 1
    else:
      histogram[0] += 1

  return (histogram / no_atoms, bin_size, no_bins, cutoff_bin)

'''
Finds the energy distribution of a system, accounting for all atoms without normalization or binning to a histogram

atom_positions: list of atom positions
atom_numbers: list of atomic numbers
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

returns: an array with the potential energy values for all atoms
'''
def find_energy_distribution_raw(no_atoms, atom_positions, atom_numbers, box_length, energy_fxn, energy_params):
  energy_array = []

  for i in range(no_atoms):
    energy = single_energy(atom_numbers, atom_positions, no_atoms, box_length, i, energy_fxn, energy_params)
    energy_array.append(energy)

  return energy_array

In [12]:
# @title Iteration Helpers
@njit(fastmath = True)
def accept_or_reject(energy_difference, temperature):
  if(energy_difference < 0):
    return True
  else:
    probability = np.exp(-((energy_difference) / (temperature * kB)))
    if(np.random.rand() < probability):
      return True
    else:
      return False

@njit(fastmath = True)
def process_move_molecule(move_accepted, energy_difference, current_positions, new_potential_energy, energy_array, move_accepted_array,
                          back_pos, forward_pos, vectors, length, append, i, n_warmup):
  if(move_accepted):
    new_potential_energy += energy_difference
    move_accepted_array[i] = True
  else:
    for selected_atom in range(back_pos, forward_pos):
      vector = vectors[selected_atom - back_pos]
      current_positions[selected_atom] -= vector
      current_positions[selected_atom] %= length
    move_accepted_array[i] = False

  if(i >= n_warmup and append):
    energy_array[i - n_warmup] = new_potential_energy

  return (current_positions, new_potential_energy, energy_array, move_accepted_array)

import numpy as np
from numba import njit
'''
vector: the vector to rotate
rotation_axis: a unit vector denoting the axis of rotation
theta: the angle of rotation (radians)
'''
@njit(fastmath=True)
def rodrigues_rotation(vector, rotation_axis, theta):

    cos_t = np.cos(theta)
    sin_t = np.sin(theta)

    axis_dot_vector = np.dot(rotation_axis, vector)

    # Cross product k x v
    cross_prod = np.cross(rotation_axis, vector)

    rodrigues_x = vector[0] * cos_t + cross_prod[0] * sin_t + rotation_axis[0] * (axis_dot_vector * (1.0 - cos_t))
    rodrigues_y = vector[1] * cos_t + cross_prod[1] * sin_t + rotation_axis[1] * (axis_dot_vector * (1.0 - cos_t))
    rodrigues_z = vector[2] * cos_t + cross_prod[2] * sin_t + rotation_axis[2] * (axis_dot_vector * (1.0 - cos_t))

    return np.array([rodrigues_x, rodrigues_y, rodrigues_z])

'''
To apply the Rodrigues formula to a rotation axis not from the origin the vector must be shifted so trig applies correctly

current_position: the vector to rotate (also the current position)
rotation_center: the center of rotation
rotation_axis: a unit vector denoting the axis of rotation
theta: the angle of rotation (radians)

returns: a vector that when added to current_position, gives the new position
'''
@njit(fastmath=True)
def rotate_around_point(current_position, rotation_center, rotation_axis, theta):
    relative_vector = current_position - rotation_center
    rotated_vector = rodrigues_rotation(relative_vector, rotation_axis, theta)
    new_positions = rotated_vector + rotation_center
    return new_positions - current_position

In [76]:
# @title Iteration and Move Types

'''
Documentation for global integers representing the possible move types for atoms and molecules. Each integer in the range of [0, move_types) represents a different type of move
0: atom translation

0: molecule translation
1: molecule rotation
2: bond stretching
'''
atom_move_types = 1
molecule_move_types = 3


'''
select_range: the lower and upper indices of current_positions to attempt a translation with
'''
@njit(fastmath = True)
def attempt_move_translation_atom(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                  atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                  energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                  current_positions, total_potential_energy, energy_array, move_accepted_array, append):

  current_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                                 no_atoms, length, selected_atom, back_pos, forward_pos,
                                 energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)

  vector = np.random.normal(0, step_size, 3)
  current_positions[selected_atom] += vector
  current_positions[selected_atom] %= length

  new_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                             no_atoms, length, selected_atom, back_pos, forward_pos,
                             energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)

  energy_difference = new_energy - current_energy
  new_potential_energy = total_potential_energy

  move_accepted = accept_or_reject(energy_difference, temperature)
  if(move_accepted):
    new_potential_energy += energy_difference
    move_accepted_array[i] = True
  else:
    current_positions[selected_atom] -= vector
    current_positions[selected_atom] %= length
    move_accepted_array[i] = False

  if(i >= n_warmup and append):
    energy_array[i - n_warmup] = new_potential_energy

  return (current_positions, new_potential_energy, energy_array, move_accepted_array)

'''
select_range: the lower and upper indices of current_positions to attempt a translation with
'''
@njit(fastmath = True)
def attempt_move_translation_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                      atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                      energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                      current_positions, total_potential_energy, energy_array, move_accepted_array, append):

  molecule_range = range(back_pos, forward_pos)
  vector = np.random.normal(0, step_size, 3)
  total_current_energy = 0

  for selected_atom in molecule_range:
    current_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                                   no_atoms, length, selected_atom, back_pos, forward_pos,
                                   energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)
    total_current_energy += current_energy

  vectors = np.zeros((forward_pos - back_pos, 3))
  for selected_atom in molecule_range:
    current_positions[selected_atom] += vector
    current_positions[selected_atom] %= length
    vectors[selected_atom - back_pos] = vector

  total_new_energy = 0
  for selected_atom in molecule_range:
    new_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                               no_atoms, length, selected_atom, back_pos, forward_pos,
                               energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)
    total_new_energy += new_energy

  energy_difference = total_new_energy - total_current_energy
  new_potential_energy = total_potential_energy

  move_accepted = accept_or_reject(energy_difference, temperature)
  (current_positions, new_potential_energy, energy_array, move_accepted_array) = process_move_molecule(move_accepted, energy_difference, current_positions, new_potential_energy, energy_array, move_accepted_array,
                                                                                       back_pos, forward_pos, vectors, length, append, i, n_warmup)

  return (current_positions, new_potential_energy, energy_array, move_accepted_array)

@njit(fastmath = True)
def attempt_move_rotation_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                   atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                   energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                   current_positions, total_potential_energy, energy_array, move_accepted_array, append):

  molecule_range = range(back_pos, forward_pos)
  rotation_vector = unit() # a unit vector describing the axis of rotation
  rotation_angle = np.random.uniform(0, 2 * np.pi) # not sure what to put for this
  total_current_energy = 0

  for selected_atom in molecule_range:
    current_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                                   no_atoms, length, selected_atom, back_pos, forward_pos,
                                   energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)
    total_current_energy += current_energy

  rotation_point = np.zeros(3)
  for selected_atom in molecule_range:
    rotation_point += current_positions[selected_atom]
  rotation_point /= (forward_pos - back_pos)

  vectors = np.zeros((forward_pos - back_pos, 3))
  for selected_atom in molecule_range:
    vector = rotate_around_point(current_positions[selected_atom], rotation_point, rotation_vector, rotation_angle)
    current_positions[selected_atom] += vector
    current_positions[selected_atom] %= length
    vectors[selected_atom - back_pos] = vector

  total_new_energy = 0
  for selected_atom in molecule_range:
    new_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                               no_atoms, length, selected_atom, back_pos, forward_pos,
                               energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)
    total_new_energy += new_energy

  energy_difference = total_new_energy - total_current_energy
  new_potential_energy = total_potential_energy

  move_accepted = accept_or_reject(energy_difference, temperature)
  result = process_move_molecule(move_accepted, energy_difference, current_positions, new_potential_energy, energy_array, move_accepted_array,
                                 back_pos, forward_pos, vectors, length, append, i, n_warmup)
  (current_positions, new_potential_energy, energy_array, move_accepted_array) = result
  if(i >= n_warmup and append):
    energy_array[i - n_warmup] = new_potential_energy

  return (current_positions, new_potential_energy, energy_array, move_accepted_array)


'''
Moves part of a molecule recursively. ALL atoms are moved in the same direction
'''
@njit(fastmath = True)
def propagate_move(selected_child, direct_children_array, current_positions, move_vector, length):
  current_positions[selected_child] += move_vector
  current_positions[selected_child] %= length

  updated_positions = current_positions
  for child in direct_children_array[selected_child]:
    if(child < 0):
      break
    updated_positions = propagate_move(child, direct_children_array, updated_positions, move_vector, length)

  return updated_positions

'''
Stretches a bond by moving selecting one branch of the molecule and moving all atoms in it in the same direction as the bond
  Case 1: the selected atom has a parent ==> the selected atom moves according to its bond with the parent
  Case 2: the selected atom does not have a parent ==> the selected atom stays still, but one of its children moves in the same direction as the bond

'''
@njit(fastmath = True)
def attempt_move_bond_stretch_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                       atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                       energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size, bond_step_size,
                                       current_positions, total_potential_energy, energy_array, move_accepted_array, append):

  selected_position = current_positions[selected_atom]
  move_vector = np.zeros(3)
  parent = direct_parent_array[selected_atom]
  move_magnitude = np.random.normal(0, bond_step_size)

  current_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                                 no_atoms, length, selected_atom, back_pos, forward_pos,
                                 energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)

  if(parent == -1):
    # there is no parent, so a child is picked at random to move
    no_children = len(direct_children_array[selected_atom])
    random_child = np.random.randint(0, no_children)
    selected_child = direct_children_array[selected_atom][random_child]

    bond_vector = current_positions[selected_child] - selected_position
    bond_vector_magnitude = np.linalg.norm(bond_vector)
    bond_vector_unit = bond_vector / bond_vector_magnitude
    move_vector = bond_vector_unit * move_magnitude

    propagate_move(selected_child, direct_children_array, current_positions, move_vector, length)

  else:
    parent_position = current_positions[parent]

    bond_vector = selected_position - parent_position
    bond_vector_magnitude = np.linalg.norm(bond_vector)
    bond_vector_unit = bond_vector / bond_vector_magnitude
    move_vector = bond_vector_unit * move_magnitude

    propagate_move(selected_atom, direct_children_array, current_positions, move_vector, length)

  new_energy = single_energy(atom_numbers, current_positions, direct_children_array, direct_parent_array,
                             no_atoms, length, selected_atom, back_pos, forward_pos,
                             energy_fxn, energy_params, bond_energy_fxn, bond_energy_params)

  energy_difference = new_energy - current_energy
  new_potential_energy = total_potential_energy

  move_accepted = accept_or_reject(energy_difference, temperature)
  # because we don't know in advance which atoms or how many atoms will be moved we have to do post-processing here
  if(move_accepted):
    new_potential_energy += energy_difference
    move_accepted_array[i] = True
  else:
    move_accepted_array[i] = False
    if(parent == -1):
      propagate_move(selected_child, direct_children_array, current_positions, -move_vector, length)
    else:
      propagate_move(selected_atom, direct_children_array, current_positions, -move_vector, length)

  return (current_positions, new_potential_energy, energy_array, move_accepted_array)

'''
Iterates the Monte Carlo simulation according to the Metropolis algorithm

no_atoms: total number of atoms
atom_numbers: list of atomic numbers
current_positions: list of atomic positions
molecule_index_array = [] # [[-1, -1], [-1, -1], [2, 4], [2, 4], [2, 4]] ==> the first two atoms are singleton atoms. Atoms 3-5 are part of the same molecule
molecule_object_array: holds either None or molecule_trie objects. Useful for determining the connectivity of atoms.

length: length of the cube
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, energy_params)
energy_params: any additional parameters the user-specified energy function needs
step_size: the size of the step to take
n_warmup: the number of iterations to take before energy values are recorded for heat capacity
energy_array: an array to append total system energy values to
temperature: temperature (K)

i: the current iteration number
total_potential_energy: the total potential energy of the system
move_accepted_array: an array to append whether or not a move was accepted
append: whether or not to append to the energy array

returns: the updated positions, total potential energy, energy array (iucluding the updated total potential energy), and move accepted array

The possible types of moves:
  Translation
  Molecule Rotation
  Bond Stretching
'''
@njit(fastmath = True)
def iterate(no_atoms, length, temperature, n_warmup, i,
            atom_numbers, molecule_index_array, direct_children_array, direct_parent_array,
            energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size, bond_step_size,
            current_positions, total_potential_energy, energy_array, move_accepted_array, append):

  selected_atom = np.random.randint(0, no_atoms)
  back_pos = molecule_index_array[selected_atom][0]
  forward_pos = molecule_index_array[selected_atom][1]

  #print(f"Selected: {selected_atom}")

  if(back_pos == -1):
    is_molecule, back_pos, forward_pos = False, selected_atom, selected_atom
  else:
    is_molecule = True

  # continue here: nagivate both atom and molecule cases
  if(is_molecule):
    move_type = np.random.randint(0, molecule_move_types)
    if(move_type == 0): # molecule translation

      #print("Case 0")
      move_result = attempt_move_translation_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                                      atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                                      energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                                      current_positions, total_potential_energy, energy_array, move_accepted_array, append)
      # move result = current_positions, new_potential_energy, energy_array, move_accepted_array

    elif(move_type == 1): # molecule rotation
      #print("Case 1")
      move_result = attempt_move_rotation_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                                   atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                                   energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                                   current_positions, total_potential_energy, energy_array, move_accepted_array, append)

    elif(move_type == 2): # bond stretching

      #print("Case 2")
      move_result = attempt_move_bond_stretch_molecule(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                                       atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                                       energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size, bond_step_size,
                                                       current_positions, total_potential_energy, energy_array, move_accepted_array, append)

    return move_result

  else:
    #print("Case 3")
    # atom translation is the only move type for atoms
    move_result = attempt_move_translation_atom(no_atoms, length, temperature, n_warmup, i, selected_atom,
                                                atom_numbers, direct_children_array, direct_parent_array, back_pos, forward_pos,
                                                energy_fxn, energy_params, bond_energy_fxn, bond_energy_params, step_size,
                                                current_positions, total_potential_energy, energy_array, move_accepted_array, append)

  return move_result

In [72]:
# @title Main Function
'''
The main function which implmentes the Monte Carlo simulation according to the Metropolis algorithm
Volume, number of atoms, and temperature are fixed. Periodic boundary conditions are used.

density: number density of the atoms/molecules (object / A^3)
no_atoms: total number of objects (NOT the total numbers of atoms)
species: list of tuples of the type (Atomic Symbol, Mol Fraction, Object Type). For correctness, this list must be unique
iterations: the mnaximum number of iterations the  simulation will complete
temperature: temperature (K)

energy: a function to calculate pairwise interaction energies between atoms of different molecules. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_derivative: a function to calculate the pairwise interaction energy derivative. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_params: any additional parameters the user-specified energy function needs

bond_energy: a function to calculate pairwise interaction energies between atoms of the same molecule. Must be of the form fxn(radius, atom#1, atom#2, params)
bond_energy_derivative: a function to calculate the pairwise interaction energy derivative. Must be of the form fxn(radius, atom#1, atom#2, params)
bond_params: any additional parameters the user-specified
tail_correction_fxn: a function to calculate the tail correction. Must be of the form fxn(rho1, rho2, atom#1, atom#2, params, cutoff_radius)

n_warmup: after this many iterations, the simulation is assumed to be at equilibrium
display_energy_profile: whether to sample the system throughout the simulation to make a distribution of each atom's potential energy value (warning: this will make the simulation run a lot slower)
cube_start: whether to start the atoms in a cubic lattice structure
anneal: whether to apply annealing at the start of the simulation
reference_start: whether to use the equilibrium reference positions

MC_initialized: if this is None, initialize the system as normal. Otherwise, start from the positions given in MC_initialized (careful: try not to change other parameters between calls)
skip_final: whether to skip the final RDF samples

move_step_size: the variance of the translational moves (A^2)
bond_step_size: the variance of the bond stretching moves (A^2)
'''
# consdier making step_size a tuple or array for the different types of moves
def MC_main(density, no_objects, species, iterations, temp,
            energy = lennard_jones, energy_derivative = lennard_jones_derivative, energy_params = LJ_avg_array,
            bond_energy = harmonic_bond,  bond_energy_derivative = harmonic_bond_derivative, bond_params = harmonic_bond_array,
            tail_correction_fxn = frenkel_smit_tail_correction,
            n_warmup = 100000, display_energy_profile = False, cube_start = False, anneal = False, reference_start = False,
            MC_initialized = None, skip_final = False,
            move_step_size = 0.400, bond_step_size = 0.100):

  energy_fxn = energy
  if(MC_initialized is None):
    MC_tuple = MC_initial(no_objects, density, species, cube_start, reference_start)
  else:
    MC_tuple = MC_initialized

  MC, molecule_index_array, molecule_object_array, direct_children_array, direct_parent_array = MC_tuple
  no_atoms = len(molecule_index_array)
  length = density_to_length(no_objects, density)

  r_cutoff = length / 2.01
  default_step_size = move_step_size
  step_size = default_step_size

  atom_names = MC.get_chemical_symbols()
  atom_numbers = MC.get_atomic_numbers()
  atom_number_fraction_list = generate_atomic_fractions(atom_numbers) # index == atomic number, entry == fraction (in terms of total atoms in the system)

  frames = []
  energy_array = np.zeros(iterations - n_warmup)
  current_position = MC.get_positions()

  # the simulation will start annealing at this temperature whenever annealing begins
  anneal_temp = temp + 300
  anneal_temp_step_size = 0.90 # whenever a step of annealing is finished, decrease the temperature by this factor (exponential cooling)
  given_temp = temp # annealing will end once we reach this temperature
  anneal_iterations = 2.5e4 # number of iterations per stage of annealing
  anneal_step_size = 1.00 * step_size # testing if larger step sizes help annealing

  currently_annealing = anneal # whether the simulation is currently annealing
  current_anneal_iteration = 0
  current_anneal_temperature = anneal_temp

  if(anneal):
    temperature = anneal_temp
    current_anneal_iterations = anneal_iterations
    step_size = anneal_step_size
  else:
    temperature = temp

  # this logic determines how the final iterations to find RDF averages and pressure averages are spaced
  if(skip_final):
    final_samples = 1
    final_sparse = 1
  else:
    final_samples = 25 # the number of final samples to take for the RDFs
    final_sparse = 1000 # how many interations to go through between each final sample
  final_iterations = final_samples * final_sparse
  no_bins = int(7.5 * length)

  total_potential_energy = total_energy(atom_numbers, current_position, direct_children_array, direct_parent_array, molecule_index_array,
                                        no_atoms, length,
                                        energy_fxn, energy_params, bond_energy, bond_params)

  # these hardcoded constants are for a specific module (energy distribution as the simulation progresses)
  energy_profile_sparse = 500 # how often to sample the rest of the energy profiles
  energy_profile_array_size = int(iterations / energy_profile_sparse)
  energy_profile_array = np.zeros(energy_profile_array_size, dtype = object)
  energy_profile_bin_size = 7.5e-2 # the size of each bin on the histogram
  positive_bins = 4 # the number of bins representing positive potential energies
  negative_bins = 16 # the number of bins representing negative potential energies
  energy_profile_bin_number = positive_bins + negative_bins
  energy_profile_index = 0

  move_accepted_array = np.zeros(iterations, dtype = bool)

  # iterates over ther user-specified number of iterations
  for i in tqdm(range(iterations), leave = False, desc = "Main Iterations: "):
    current_position, total_potential_energy, energy_array, move_accepted_array = iterate(no_atoms, length, temperature, n_warmup, i,
                                                                                          atom_numbers, molecule_index_array, direct_children_array, direct_parent_array,
                                                                                          energy_fxn, energy_params, bond_energy, bond_params, step_size, bond_step_size,
                                                                                          current_position, total_potential_energy, energy_array, move_accepted_array, False)

    # writing to OVITO
    if i % 50 == 0:
      MC.set_positions(current_position)
      frame = MC.copy()
      frame.info["step"] = i
      frame.info["temperature"] = temperature
      frames.append(frame)

    if(display_energy_profile):
      if(i % energy_profile_sparse == 0 and energy_profile_index < energy_profile_array_size):
        energy_profile_array[energy_profile_index] = find_energy_distribution(energy_profile_bin_size, energy_profile_bin_number, negative_bins, no_atoms, current_position, atom_numbers, length,
                                                                              energy_fxn, energy_params)
        energy_profile_index += 1

    if(currently_annealing):
      current_anneal_iteration -= 1

      if(current_anneal_iteration < 0):
        current_anneal_iteration = anneal_iterations
        current_anneal_temperature *= anneal_temp_step_size

        if(current_anneal_temperature < given_temp):
          currently_annealing = False
          temperature = given_temp
          step_size = default_step_size
        else:
          temperature = current_anneal_temperature

  unique_species = (pd.unique(pd.Series(atom_names)))
  unique_numbers = (pd.unique(pd.Series(atom_numbers)))
  len_species = len(unique_species)

  R_bins = bins_to_distance(length, no_bins)
  raw_RDFs = np.empty((final_samples, cantor_pair(len_species, len_species)), dtype = object)
  virial_pressure_array = []

  for final in tqdm(range(final_iterations), leave = False, desc = "Finding RDFs: "):
    current_position, total_potential_energy, energy_array, move_accepted_array = iterate(no_atoms, length, temperature, n_warmup, 0,
                                                                                          atom_numbers, molecule_index_array, direct_children_array, direct_parent_array,
                                                                                          energy_fxn, energy_params, bond_energy, bond_params, step_size, bond_step_size,
                                                                                          current_position, total_potential_energy, energy_array, move_accepted_array, False)

    # only take a smple RDF every final_sparse iterations
    if(final % final_sparse == 0):
      index = final // final_sparse
      for i in range(len_species):
        for j in range(len_species):
          species1 = unique_species[i]
          species2 = unique_species[j]

          RDF = partial_RDF(species1, species2, atom_names, no_atoms, current_position, length, R_bins)
          raw_RDFs[index][cantor_pair(i, j)] = RDF

      virial_pressure_term = virial_pressure(atom_numbers, current_position, length, energy_fxn, energy_params)
      virial_pressure_total = density * kB * temperature + virial_pressure_term
      virial_pressure_array.append(virial_pressure_total)

  # calculating the RDF for each atom-atom pair
  averaged_species_RDFs = np.zeros(cantor_pair(len_species, len_species), dtype = object)
  species_RDF_names = np.zeros(cantor_pair(len_species, len_species), dtype = 'U12')

  for i in range(len_species):
    for j in range(len_species):
      sum = np.zeros(no_bins)
      for k in range(final_samples):
        sum += raw_RDFs[k][cantor_pair(i, j)]
      averaged_species_RDFs[cantor_pair(i, j)] = sum / final_samples
      species_RDF_names[cantor_pair(i, j)] = f"{unique_species[i]}{unique_species[j]}"

  # calculating the average RDF, weighted by the fraction of each atom present
  manual_RDF = np.zeros(len(averaged_species_RDFs[cantor_pair(0, 0)]))

  for i in range(len_species):
    for j in range(len_species):
      number1 = unique_numbers[i]
      number2 = unique_numbers[j]
      fraction = atom_number_fraction_list[number1] * atom_number_fraction_list[number2]
      manual_RDF += fraction * averaged_species_RDFs[cantor_pair(i, j)]


  mayer_species_RDFs = np.zeros(cantor_pair(len_species, len_species), dtype = object)
  mayer_RDF = np.zeros(no_bins)

  for i in range(len_species):
    for j in range(len_species):
      number1 = unique_numbers[i]
      number2 = unique_numbers[j]

      mayer_species_temp_array = np.zeros(no_bins)
      for r in range(no_bins):
        radius = R_bins[r]
        mayer_species_temp_array[r] = (mayer_f_fxn(radius, temperature, number1, number2, energy_params, energy_fxn))

      mayer_species_RDFs[cantor_pair(i, j)] = mayer_species_temp_array
      mayer_RDF += mayer_species_temp_array * atom_number_fraction_list[number1] * atom_number_fraction_list[number2]

  '''
  Begin pressure calculations
  '''

  RDF_pressure_term = 0
  mayer_pressure_term = 0

  for i in range(len_species):
    for j in range(i, len_species):

      number1 = unique_numbers[i]
      number2 = unique_numbers[j]
      fraction1 = atom_number_fraction_list[number1]
      fraction2 = atom_number_fraction_list[number2]

      RDF_integral_term = -fraction1 * fraction2 * density * density * pressure_equation(temperature, number1, number2, averaged_species_RDFs[cantor_pair(i, j)], R_bins, energy_params,
                                                                                     energy_fxn = energy, energy_fxn_derivative = energy_derivative, mayer = False)

      mayer_integral_term = -fraction1 * fraction2 * density * density * pressure_equation(temperature, number1, number2, averaged_species_RDFs[cantor_pair(i, j)], R_bins, energy_params,
                                                                                     energy_fxn = energy, energy_fxn_derivative = energy_derivative, mayer = True)

      tail_term = -tail_correction(tail_correction_fxn, number1, number2, fraction1, fraction2, energy_params, r_cutoff, density)
      RDF_pressure_term += RDF_integral_term + tail_term
      mayer_pressure_term += mayer_integral_term + tail_term

  RDF_pressure_total = density * kB * temperature - 2 / 3 * np.pi * RDF_pressure_term
  mayer_pressure_total = density * kB * temperature - 2 / 3 * np.pi * mayer_pressure_term
  virial_pressure_average = np.average(virial_pressure_array)

  #ASE RDF (not species specific)
  #RDF, R_bins_ASE = get_rdf(MC, length / 2.01, nbins = no_bins)
  RDF, R_bins_ASE = get_rdf(MC, length / 2.01, nbins = no_bins)

  write(f"output.extxyz", frames)
  MC.set_positions(current_position)

  MC_return_tuple = (MC, molecule_index_array, molecule_object_array, direct_children_array, direct_parent_array)

  if(display_energy_profile):
    return (MC_return_tuple, (RDF, manual_RDF, species_RDF_names, averaged_species_RDFs, mayer_RDF, mayer_species_RDFs), R_bins, no_bins, energy_profile_array, energy_array, (RDF_pressure_total, virial_pressure_average, mayer_pressure_total), move_accepted_array)
    # energy_profile_array: histogram, bin_size, no_bins, cutoff_bin
  # note: MC contains the positions of the atoms
  return (MC_return_tuple, (RDF, manual_RDF, species_RDF_names, averaged_species_RDFs, mayer_RDF, mayer_species_RDFs), R_bins, no_bins, energy_array, (RDF_pressure_total, virial_pressure_average, mayer_pressure_total), move_accepted_array)


# <UI> Learning Modules

In [15]:
# @title Constructing Prototype Polymers and Molecules

'''
Given a repeat unit and polymer length, returns a molecule_trie representing the polymer
WARNING: right now this only works with monoatomic polymers. Full detail will be implemented later

repeat_unit: a prototype molecule_trie representing one monomer
length: the length of the polymer (number of monomers)
head: whether this is the first unit of the polymer

returns: a molecule_trie representing the polymer
'''
def construct_polymer(repeat_unit, length, head):
  if(length == 1):
    return deepcopy(repeat_unit)
  assert(length >= 1)

  segment = deepcopy(repeat_unit)
  if(head):
    segment.prev = None

  next_unit = construct_polymer(repeat_unit, length - 1, False)
  segment.children.append(next_unit)
  next_unit.prev = segment

  return segment

Uranium_Monomer = molecule_trie()
Uranium_Monomer.init(92, 'U', None, True, False)

Uranium_Dimer = construct_polymer(Uranium_Monomer, 2, True)
assert(isinstance(Uranium_Dimer, molecule_trie))

Aluminium_Monomer = molecule_trie()
Aluminium_Monomer.init(13, 'Al', None, True, False)

Aluminium_Dimer = construct_polymer(Aluminium_Monomer, 2, True)
assert(isinstance(Aluminium_Dimer, molecule_trie))

Aluminium_Trimer = construct_polymer(Aluminium_Monomer, 3, True)
assert(isinstance(Aluminium_Trimer, molecule_trie))

In [16]:
# @title Module: Effect of Density and Number of Atoms on RDF Shape

fraction_list = [(Aluminium_Trimer, .50, 'Molecule'), ('Ar', 0.50, 'Atom')]
#fraction_list = [(Aluminium_Trimer, 1, 'Molecule')]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''

from line_profiler import LineProfiler

def run_MC(null_filler = None):
  clear_output()
  print("Running MC Simulation")

  Ns = np.linspace(N_slider.value[0], N_slider.value[1], atom_sweep_slider.value)
  for i in range(len(Ns)):
    Ns[i] = int(Ns[i])

  rho_lower = rho_slider.value[0]
  rho_upper = rho_slider.value[1]
  rho_sweep = rho_sweep_slider.value
  rhos_raw = np.linspace(rho_lower, rho_upper, rho_sweep) # mols / L
  rhos_simulation_units = []
  for i in range(len(rhos_raw)):
    rhos_simulation_units.append(rhos_raw[i] * 6.022e23 * 1000 * 1e-30)

  temperature = temp_slider.value

  niterations = iteration_slider.value[1]
  nwarmup = iteration_slider.value[0]

  fig2, axs2 = plt.subplots(len(rhos_simulation_units), len(Ns), figsize = (16, 12), sharex = True, sharey = True)
  fig2.suptitle('Average RDF', fontsize = 13)
  fig2.supxlabel('Distance (A)')
  fig2.supylabel('Radial Distribution Function')
  plt.tight_layout()

  fig3, axs3 = plt.subplots(len(rhos_simulation_units), len(Ns), figsize = (16, 12), sharex = True, sharey = True)
  fig3.suptitle('Species RDFs', fontsize = 13)
  fig3.supxlabel('Distance (A)')
  fig3.supylabel('Radial Distribution Function')
  plt.tight_layout()
  for i in (range(len(rhos_simulation_units))):
    for j in range(len(Ns)):
      rho = rhos_simulation_units[i]
      rho_normal_unit = rho / 6.022e23 / 1000 * 1e30
      N = int(Ns[j])

      Result = MC_main(rho, N, fraction_list, niterations, temperature,
                       energy = lennard_jones, energy_derivative = lennard_jones_derivative, energy_params = LJ_avg_array, n_warmup = nwarmup)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, species_RDF_names, species_RDFs, mayer_total_RDF, mayer_species_RDFs = RDF

      ax_m = axs2[i, j]
      ax_m.plot(R_bins, manual_RDF, label = f"Actual: N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.plot(R_bins, mayer_total_RDF, label = f"Mayer: N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.legend()
      ax_m.set_title(f"N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_m.autoscale(enable = True, axis = 'both', tight = False)

      ax_s = axs3[i, j]
      for a in range (len(fraction_list)):
        for b in range(a, len(fraction_list)):
          index = cantor_pair(a, b)
          ax_s.plot(R_bins, species_RDFs[index], label = f"Actual: {species_RDF_names[cantor_pair(a, b)]}")
          ax_s.plot(R_bins, mayer_species_RDFs[index], label = f"Mayer: {species_RDF_names[cantor_pair(a, b)]}")

      ax_s.set_title(f"N = {N}, rho = {rho_normal_unit:.3f} M")
      ax_s.autoscale(enable = True, axis = 'both', tight = False)
      ax_s.legend()

  plt.show()
  rebuild_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

N_slider = widgets.IntRangeSlider(
    value = [200, 300],        # initial range [min_selected, max_selected]
    min = 100,               # minimum possible value
    max = 2000,              # maximum possible value
    step = 100,
    description = 'Atoms:',
    style = style,
    layout = Layout(width = width)
)

rho_slider = widgets.FloatRangeSlider(
    value = [25, 75],        # initial range [min_selected, max_selected]
    min = 1,               # minimum possible value
    max = 100,                  # maximum possible value
    step = 1,
    description = 'Density:',
    style = style,
    layout = Layout(width = width)
)

atom_sweep_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Atom Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

rho_sweep_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Density Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

iteration_slider = widgets.IntRangeSlider(
    value = [0e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_objects():
  display(button)
  display(N_slider)
  display(atom_sweep_slider)
  display(rho_slider)
  display(rho_sweep_slider)
  display(temp_slider)
  display(iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
button.on_click(run_MC)

rebuild_objects()

Button(description='Run MC Simulation', icon='check', layout=Layout(width='400px'), style=ButtonStyle(), toolt…

IntRangeSlider(value=(200, 300), description='Atoms:', layout=Layout(width='400px'), max=2000, min=100, step=1…

IntSlider(value=2, continuous_update=False, description='Atom Sweep:', layout=Layout(width='400px'), max=4, mi…

FloatRangeSlider(value=(25.0, 75.0), description='Density:', layout=Layout(width='400px'), min=1.0, step=1.0, …

IntSlider(value=2, continuous_update=False, description='Density Sweep:', layout=Layout(width='400px'), max=4,…

IntSlider(value=200, continuous_update=False, description='Temperature:', layout=Layout(width='400px'), max=60…

IntRangeSlider(value=(100000, 100000), description='Iterations:', layout=Layout(width='400px'), max=10000000, …

In [17]:
# @title Performance (AI)
import cProfile
import pstats
import io
import time


def run_MC_AI(null_filler=None):
    clear_output()
    print("Running MC Simulation")

    # --- Setup (your existing code) ---
    Ns = np.linspace(N_slider.value[0], N_slider.value[1], atom_sweep_slider.value)
    for i in range(len(Ns)):
        Ns[i] = int(Ns[i])

    rho_lower = rho_slider.value[0]
    rho_upper = rho_slider.value[1]
    rho_sweep = rho_sweep_slider.value
    rhos_raw = np.linspace(rho_lower, rho_upper, rho_sweep)
    rhos_simulation_units = [r * 6.022e23 * 1000 * 1e-30 for r in rhos_raw]

    temperature = temp_slider.value
    niterations = 50000
    nwarmup = 0

    # --- Profiling start ---
    profiler = cProfile.Profile()
    profiler.enable()
    sim_times = []


    # --- Main loops ---
    for i in range(len(rhos_simulation_units)):
        for j in range(len(Ns)):
            rho = rhos_simulation_units[i]
            rho_normal_unit = rho / 6.022e23 / 1000 * 1e30
            N = int(Ns[j])

            t0 = time.perf_counter()
            Result = MC_main(rho, N, fraction_list, niterations, temperature,
                           energy=lennard_jones, energy_derivative=lennard_jones_derivative,
                           energy_params=LJ_avg_array, n_warmup=nwarmup, skip_final = True)
            t1 = time.perf_counter()
            sim_times.append((N, rho_normal_unit, t1-t0, niterations))

            MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
            ASE_RDF, manual_RDF, names, species_RDFs, mayer_total_RDF, mayer_species_RDFs = RDF

    # --- Profiling end ---
    profiler.disable()

    # Print profiling results
    s = io.StringIO()
    ps = pstats.Stats(profiler, stream=s).sort_stats('cumtime')
    ps.print_stats(25)

    print(f"\n{'='*70}")
    print("CUMULATIVE TIME PROFILE (Top 25 functions)")
    print(f"{'='*70}")
    print(s.getvalue())

    # Print simulation timing
    print(f"\n{'='*70}")
    print("PER-SIMULATION TIMING")
    print(f"{'='*70}")
    print(f"{'N':>6} {'ρ (M)':>8} {'Time (s)':>10} {'iters':>10} {'ms/iter':>10}")
    print("-" * 50)
    for N, rho, t, iters in sim_times:
        print(f"{N:6d} {rho:8.3f} {t:10.2f} {iters:10,} {t/iters*1000:10.3f}")

    plt.show()
    rebuild_objects()

#run_MC_AI()

In [18]:
# @title Module: Effect of Temperature and Density on RDF Shape

#fraction_list = [('B', 0.33), ('O', 0.33), ('S', 0.34)]
RDF_fraction_list = [('Ar', 1)]
#fraction_list = [('Ar', 0.5), ('C', 0.5)]
'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_RDF(null_filler = None):
  clear_output()
  print("Running MC Simulation")

  N = atom_RDF_slider.value

  rho_lower = rho_RDF_slider.value[0]
  rho_upper = rho_RDF_slider.value[1]
  rho_sweep = rho_sweep_RDF_slider.value
  rhos_raw = np.linspace(rho_lower, rho_upper, rho_sweep) # mols / L
  rhos_simulation_units = []
  for i in range(len(rhos_raw)):
    rhos_simulation_units.append(rhos_raw[i] * 6.022e23 * 1000 * 1e-30)

  temp_lower = temp_RDF_slider.value[0]
  temp_upper = temp_RDF_slider.value[1]
  temp_sweep = temp_sweep_RDF_slider.value
  temps = np.linspace(temp_lower, temp_upper, temp_sweep)

  niterations = iteration_RDF_slider.value[1]
  nwarmup = iteration_RDF_slider.value[0]

  fig2, axs2 = plt.subplots(len(rhos_simulation_units), len(temps), figsize = (16, 12), sharex = True, sharey = True)
  fig2.suptitle('Average RDF', fontsize = 13)
  fig2.supxlabel('Distance (A)')
  fig2.supylabel('Radial Distribution Function')
  plt.tight_layout()

  fig3, axs3 = plt.subplots(len(rhos_simulation_units), len(temps), figsize = (16, 12), sharex = True, sharey = True)
  fig3.suptitle('Species RDFs', fontsize = 13)
  fig3.supxlabel('Distance (A)')
  fig3.supylabel('Radial Distribution Function')
  plt.tight_layout()
  for i in (range(len(rhos_simulation_units))):
    for j in range(len(temps)):
      rho = rhos_simulation_units[i]
      rho_normal_unit = rho / 6.022e23 / 1000 * 1e30
      temperature = temps[j]

      Result = MC_main(rho, N, RDF_fraction_list, niterations, temperature,
                       energy = lennard_jones, energy_derivative = lennard_jones_derivative, energy_params = LJ_avg_array, n_warmup = nwarmup)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, species_RDFs, mayer_total_RDF, mayer_species_RDFs = RDF

      ax_m = axs2[i, j]
      ax_m.plot(R_bins, manual_RDF, label = f"Actual: rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.plot(R_bins, mayer_total_RDF, label = f"Mayer: rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.legend()
      ax_m.set_title(f"rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_m.autoscale(enable = True, axis = 'both', tight = False)

      ax_s = axs3[i, j]
      for a in range (len(fraction_list)):
        for b in range(a, len(fraction_list)):
          index = cantor_pair(a, b)
          ax_s.plot(R_bins, species_RDFs[index], label = f"Actual: {fraction_list[a][0]}{fraction_list[b][0]}")
          ax_s.plot(R_bins, mayer_species_RDFs[index], label = f"Mayer: {fraction_list[a][0]}{fraction_list[b][0]}")

      ax_s.set_title(f"rho = {rho_normal_unit:.3} M, T = {temperature:.1f} K")
      ax_s.autoscale(enable = True, axis = 'both', tight = False)
      ax_s.legend()

  plt.show()
  rebuild_objects_RDF()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

temp_RDF_slider = widgets.IntRangeSlider(
    value = [150, 350],        # initial range [min_selected, max_selected]
    min = 50,               # minimum possible value
    max = 800,              # maximum possible value
    step = 50,
    description = 'Temperature:',
    style = style,
    layout = Layout(width = width)
)

rho_RDF_slider = widgets.FloatRangeSlider(
    value = [55, 75],        # initial range [min_selected, max_selected]
    min = 1,               # minimum possible value
    max = 100,                  # maximum possible value
    step = 1,
    description = 'Density:',
    style = style,
    layout = Layout(width = width)
)

temp_sweep_RDF_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Temp Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

rho_sweep_RDF_slider = widgets.IntSlider(
    value = 2,
    min = 2,
    max = 4,
    step = 1,
    description = 'Density Sweep:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

atom_RDF_slider = widgets.IntSlider(
    value = 400,
    min = 200,
    max = 1000,
    step = 100,
    description = 'Atoms:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

iteration_RDF_slider = widgets.IntRangeSlider(
    value = [0e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_objects_RDF():
  display(button_RDF)
  display(rho_RDF_slider)
  display(rho_sweep_RDF_slider)
  display(temp_RDF_slider)
  display(temp_sweep_RDF_slider)
  display(atom_RDF_slider)
  display(iteration_RDF_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
button_RDF = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
button_RDF.on_click(run_MC_RDF)

rebuild_objects_RDF()

Button(description='Run MC Simulation', icon='check', layout=Layout(width='400px'), style=ButtonStyle(), toolt…

FloatRangeSlider(value=(55.0, 75.0), description='Density:', layout=Layout(width='400px'), min=1.0, step=1.0, …

IntSlider(value=2, continuous_update=False, description='Density Sweep:', layout=Layout(width='400px'), max=4,…

IntRangeSlider(value=(150, 350), description='Temperature:', layout=Layout(width='400px'), max=800, min=50, st…

IntSlider(value=2, continuous_update=False, description='Temp Sweep:', layout=Layout(width='400px'), max=4, mi…

IntSlider(value=400, continuous_update=False, description='Atoms:', layout=Layout(width='400px'), max=1000, mi…

IntRangeSlider(value=(100000, 100000), description='Iterations:', layout=Layout(width='400px'), max=10000000, …

In [19]:
# @title Module: Energy Distribution as Simulation Progresses

energy_fraction_list = [('He', 1.0)]

x_values = []
y_values = []
energy_display = []
has_ran = False
boltzmann_params = 0
expon_params = 0
lowest_energy = 0

def get_energy_profile(index):
  if(index < len(energy_display)):
    return energy_display[index][0]
  elif (index < 0):
    return energy_display[0][0]
  else:
    return energy_display[len(energy_display) - 1][0]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_energy(null_filler = None):
  clear_output()
  print("Running MC Simulation - Energy Distribution Module")

  rho = 0.004
  no_atoms = 800
  temperature = energy_temp_slider.value
  niterations = energy_iteration_slider.value[1]
  nwarmup = energy_iteration_slider.value[0]

  Result = MC_main(rho, no_atoms, energy_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, display_energy_profile = True, cube_start = True)
  MC, RDF, R_bins, no_bins, energy_profiles, heat_capacity, pressure, move_accepted_array = Result

  global energy_display
  energy_display = energy_profiles
  # energy_profiles: list of (histogram, bin_size, no_bins, cutoff_bin)

  first_profile = energy_display[0]
  frequency_values = first_profile[0]
  bin_size = first_profile[1]
  no_bins = first_profile[2]
  cutoff_bin = first_profile[3]

  bin_values = np.zeros(int(no_bins))
  bin_values[0] = -cutoff_bin * bin_size
  for i in range(1, no_bins):
    bin_values[i] = bin_values[i - 1] + bin_size

  global has_ran
  global x_values
  has_ran = True
  x_values = bin_values
  rebuild_energy_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

energy_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 400,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

energy_iteration_slider = widgets.IntRangeSlider(
    value = [0e5, 4e5],
    min = 0e5,               # minimum possible value
    max = 4e5,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_energy_objects():
  display(energy_button)
  display(energy_temp_slider)
  display(energy_iteration_slider)

  if(has_ran):
    display(energy_interactive_plot)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
energy_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
energy_button.on_click(run_MC_energy)

slider = IntSlider(
    value = 0,
    min = 0,
    max = 800,
    step = 1,
    description = 'Move Me',
    continuous_update = True,
    style = style,
    layout = Layout(width = width)
)

def f(i = 0):
    x = x_values
    if(has_ran):
      y = get_energy_profile(i)
    else:
      return

    xnew = np.linspace(x[0], x[-1], 300)
    spline = make_interp_spline(x, y, k = 3)
    y_smooth = spline(xnew)

    plt.plot(xnew, y_smooth, label = 'data')
    plt.ylim(0, 1.0)
    plt.xlabel('Energy (eV)')
    plt.ylabel('Relative Frequency')
    plt.title('Energy Distribution')
    plt.legend()
    plt.show()

energy_interactive_plot = interactive(f, i = slider)
output = energy_interactive_plot.children[-1]
output.layout.height = '500px'
clear_output()

rebuild_energy_objects()

Button(description='Run MC Simulation', icon='check', layout=Layout(width='400px'), style=ButtonStyle(), toolt…

IntSlider(value=200, continuous_update=False, description='Temperature:', layout=Layout(width='400px'), max=40…

IntRangeSlider(value=(0, 400000), description='Iterations:', layout=Layout(width='400px'), max=400000, readout…

In [20]:
# @title Module: Energy Distribution - Multiple Iterations

boltzmann_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_boltzmann(null_filler = None):
  clear_output()

  print("Running MC Simulation - Energy Distribution Module")

  N = 512
  rho = 0.004
  total = N
  iterations = 4

  temperature_low = boltzmann_temp_slider.value[0]
  temperature_high = boltzmann_temp_slider.value[1]
  temperatures = np.linspace(temperature_low, temperature_high, iterations)

  niterations = boltzmann_iteration_slider.value[1]
  nwarmup = boltzmann_iteration_slider.value[0]

  for temperature in temperatures:
    Result = MC_main(rho, total, boltzmann_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, cube_start = True)
    MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result

    plt.hist(heat_capacity, bins = 30, density = True, label = f'T = {temperature:.1f} K', histtype = 'step')
  plt.xlabel('Energy (eV)')
  plt.ylabel('Relative Frequency')
  plt.title('Energy Distribution')
  plt.legend()
  plt.show()

  rebuild_boltzmann_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

boltzmann_temp_slider = widgets.IntRangeSlider(
    value = [250, 1500],
    min = 100,
    max = 1600,
    step = 50,
    description = 'Temperature:',
    style = style,
    layout = Layout(width = width)
)

boltzmann_iteration_slider = widgets.IntRangeSlider(
    value = [0.5e5, 4e5],
    min = 0.5e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_boltzmann_objects():
  display(boltzmann_button)
  display(boltzmann_temp_slider)
  display(boltzmann_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
boltzmann_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
boltzmann_button.on_click(run_MC_boltzmann)

rebuild_boltzmann_objects()

Button(description='Run MC Simulation', icon='check', layout=Layout(width='400px'), style=ButtonStyle(), toolt…

IntRangeSlider(value=(250, 1500), description='Temperature:', layout=Layout(width='400px'), max=1600, min=100,…

IntRangeSlider(value=(50000, 400000), description='Iterations:', layout=Layout(width='400px'), max=10000000, m…

In [21]:
# @title Module: Obtaining Normalized Heat Capacity

heat_capacity_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_heat_capacity(null_filler = None):
  clear_output()
  print("Running MC Simulation - Heat Capacity Module")

  trials = 5
  heat_capacity_array = np.zeros(trials)

  N = 256

  raw_density = 1202.05 # kg / m^3 (Argon)
  rho = raw_density * 1000 / 39.948 * 1e-10 * 1e-10 * 1e-10 * 6.022e23 # kg to grams, mols of Argon per gram, meters to A, mols to atoms
  total = N

  temperature = 100 # K
  niterations = heat_capacity_iteration_slider.value[1]
  nwarmup = heat_capacity_iteration_slider.value[0]

  accepted_moves = np.zeros(trials)
  accepted_moves_x = np.zeros(trials)
  for i in range(len(accepted_moves)):
    accepted_moves_x[i] = i

  x_temp = np.zeros(niterations - nwarmup - 1)
  for i in range(len(x_temp)):
    x_temp[i] = i

  for i in (range(trials)):
    Result = MC_main(rho, total, heat_capacity_fraction_list, niterations, temperature, params = LJ_avg_array, n_warmup = nwarmup, reference_start = True)
    MC, RDF, R_bins, no_bins, potential_energy_values, pressure, move_accepted_array = Result

    heat_capacity = np.var(potential_energy_values) / (total * temperature * temperature * kB * kB)
    heat_capacity_array[i] = heat_capacity
    plt.plot(x_temp, potential_energy_values)

    accepted_moves[i] = np.average(move_accepted_array)

  plt.xlabel('Iterations')
  plt.ylabel('Potential Energy (eV)')
  plt.title('Potential Energy vs Iteration')
  plt.show()
  heat_capacity_average = np.average(heat_capacity_array)
  heat_capacity_error = np.std(heat_capacity_array)

  print(f"The average normalized heat capacity is {heat_capacity_average} +/- {heat_capacity_error} 1 / atom")
  plt.hist(heat_capacity_array)
  plt.xlabel('Heat Capacity (1 / atom)')
  plt.ylabel('Frequency')
  plt.title('Heat Capacity Distribution')
  plt.show()

  plt.plot(accepted_moves_x, accepted_moves)
  plt.title('Rate of Accepted Moves')
  plt.show()

  rebuild_heat_capacity_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

heat_capacity_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0,
    max = 400,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

heat_capacity_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 2e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_heat_capacity_objects():
  display(heat_capacity_button)
  #display(heat_capacity_temp_slider)
  display(heat_capacity_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
heat_capacity_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
heat_capacity_button.on_click(run_MC_heat_capacity)

rebuild_heat_capacity_objects()

Button(description='Run MC Simulation', layout=Layout(width='400px'), style=ButtonStyle(), tooltip='Run MC Sim…

IntRangeSlider(value=(100000, 200000), description='Iterations:', layout=Layout(width='400px'), max=10000000, …

In [22]:
# @title Module: Obtaining Pressure and Virial Coefficients

pressure_fraction_list = [('Ar', 1.0)]

'''
Virial expansion to fit for SciPy
density: the density of the system (mols / L)
B: the second virial coefficient  (A^3 / atom)
C: the third virial coefficient (A^6 / atom^2)

returns: the quantity P / kB * T
'''
def virial_equation_reduced(density, B, C):
  return density + (density ** 2) * B + (density ** 3) * C

'''
density: density (mols / L)
temperature: temperature (K)

Returns: the pressure (Pa)
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure(null_filler = None):
  clear_output()
  print("Running MC Simulation - Pressure Module")

  #ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  iterations = 25
  no_atoms = 256

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  rhos_normal_units = np.geomspace(0.15, 25, iterations)

  temperature = pressure_temp_slider.value
  niterations = pressure_iteration_slider.value[1]
  nwarmup = pressure_iteration_slider.value[0]

  energy_arrays = []

  rhos = []
  for density in rhos_normal_units:
    rhos.append(density * 6.022e23 * 1000 * 1e-30) # convert to atoms / A^3
    ideal_gas_law_array.append(ideal_gas_law(density, temperature))

    # convert from mols / L to m^3 / mol
    #molar_volume = (1 / density) / 1000
    #LJ_Ref_array.append(reference_pressure(ref_model, molar_volume, temperature))


  for i in range(len(rhos)):
    rho = rhos[i]
    Result = MC_main(rho, no_atoms, pressure_fraction_list, niterations, temperature, n_warmup = nwarmup, reference_start = True,
                    params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
    MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

    RDF_pressure, virial_pressure, mayer_pressure = pressure
    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(rhos_normal_units, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(rhos_normal_units, result_pressure_virial_normal_units, label = 'Virial')
  plt.scatter(rhos_normal_units, result_pressure_mayer_normal_units, label = 'Mayer')
  #plt.plot(rhos_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.plot(rhos_normal_units, ideal_gas_law_array, label = 'Ideal Gas Law')
  plt.xlabel('Density (mol / L)')
  plt.ylabel('Pressure (Pa)')
  plt.title(f'Pressure vs Density - Data - T = {temperature}')
  plt.legend()
  plt.show()

  pressure_over_kBT_RDF = RDF_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_RDF)
  B_RDF, C_RDF = popt
  perr = np.sqrt(np.diag(pcov))

  pressure_over_kBT_viral = virial_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_viral)
  B_virial, C_virial = popt
  perr = np.sqrt(np.diag(pcov))

  pressure_over_kBT_mayer = mayer_pressure_array / (kB * temperature)
  popt, pcov = curve_fit(virial_equation_reduced, rhos, pressure_over_kBT_mayer)
  B_mayer, C_mayer = popt
  perr = np.sqrt(np.diag(pcov))

  fit_pressures_RDF = []
  fit_pressures_virial = []
  fit_pressures_mayer = []

  for i in range(len(rhos)):
    rho = rhos[i]
    fit_pressures_RDF.append(virial_equation_reduced(rho, B_RDF, C_RDF) * (kB * temperature))
    fit_pressures_virial.append(virial_equation_reduced(rho, B_virial, C_virial) * (kB * temperature))
    fit_pressures_mayer.append(virial_equation_reduced(rho, B_mayer, C_mayer) * (kB * temperature))

  pressures_normal_units_RDF = []
  pressures_normal_units_virial = []
  pressures_normal_units_mayer = []

  for i in range(len(fit_pressures_RDF)):
    pressures_normal_units_RDF.append(fit_pressures_RDF[i] * 1.602e-19 * 1e30) # pascals
    pressures_normal_units_virial.append(fit_pressures_virial[i] * 1.602e-19 * 1e30) # pascals
    pressures_normal_units_mayer.append(fit_pressures_mayer[i] * 1.602e-19 * 1e30) # pascals

  plt.scatter(rhos_normal_units, pressures_normal_units_RDF, label = 'RDF Fit')
  plt.scatter(rhos_normal_units, pressures_normal_units_virial, label = 'Virial Fit')
  plt.scatter(rhos_normal_units, pressures_normal_units_mayer, label = 'Mayer Fit')
  #plt.plot(rhos_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.plot(rhos_normal_units, ideal_gas_law_array, label = 'Ideal Gas Law')
  plt.xlabel('Density (mol / L)')
  plt.ylabel('Pressure (Pa)')
  plt.title(f'Pressure vs Density - Fit Equations - T = {temperature}')
  plt.legend()
  plt.show()

  print(f"RDF - The second virial coefficient is {B_RDF * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"RDF - The third vir ial coefficient is {C_RDF * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")
  print(f"Virial - The second virial coefficient is {B_virial * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"Virial - The third virial coefficient is {C_virial * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")
  print(f"Mayer - The second virial coefficient is {B_mayer * 1e-30 * 6.022e23} +/- {perr[0]} m^3 / mol")
  print(f"Mayer - The third virial coefficient is {C_mayer * 1e-60 * 6.022e23 * 6.022e23} +/- {perr[0]} m^6 / mol^2")

  rebuild_pressure_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_slider = widgets.IntSlider(
    value = 200,
    min = 0.1,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

pressure_iteration_slider = widgets.IntRangeSlider(
    value = [0.1e5, 1.5e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_objects():
  display(pressure_button)
  display(pressure_temp_slider)
  display(pressure_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_button.on_click(run_MC_pressure)

rebuild_pressure_objects()

Button(description='Run MC Simulation', layout=Layout(width='400px'), style=ButtonStyle(), tooltip='Run MC Sim…

IntSlider(value=200, continuous_update=False, description='Temperature:', layout=Layout(width='400px'), max=60…

IntRangeSlider(value=(10000, 150000), description='Iterations:', layout=Layout(width='400px'), max=10000000, r…

In [23]:
# @title Module: Critical Point

pressure_fraction_list = [('Ar', 1.0)]


'''
Calculates the pressure according to ideal gas law
density: density (mols / L)
temperature: temperature (K)

Returns: the pressure in pascals
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000


'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure(null_filler = None):
  clear_output()
  print("Running MC Simulation - Pressure Module")

  #ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  iterations = 10
  no_atoms = 256

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  molar_volume_normal_units = np.linspace(4e-9, 150, iterations) # m^3 / mol

  temperature = pressure_temp_slider.value
  niterations = pressure_iteration_slider.value[1]
  nwarmup = pressure_iteration_slider.value[0]

  energy_arrays = []
  molar_vols = []
  for spec_vol in molar_volume_normal_units:
    molar_vols.append(spec_vol / 6.022e23 * 1e30) # convert to A^3 / atom
    ideal_gas_law_array.append(ideal_gas_law(1 / spec_vol, temperature))
    #LJ_Ref_array.append(reference_pressure(ref_model, spec_vol, temperature))

  for i in range(len(molar_vols)):
    molar_vol = molar_vols[i]
    rho = 1 / molar_vol
    rho_convert = rho
    Result = MC_main(rho_convert, no_atoms, pressure_fraction_list, niterations, temperature, n_warmup = nwarmup, reference_start = True,
                    params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
    MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

    RDF_pressure, virial_pressure, mayer_pressure = pressure
    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(molar_volume_normal_units, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(molar_volume_normal_units, result_pressure_virial_normal_units, label = 'Virial')
  #plt.plot(molar_volume_normal_units, LJ_Ref_array, label = 'Reference LJ')
  plt.xlabel('Molar Volume (m^3 / mol)')
  plt.ylabel('Pressure (Pa)')
  plt.yscale('log')
  plt.title(f'Pressure vs Density - Data - T = {temperature}')
  plt.legend()
  plt.show()

  rebuild_pressure_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_slider = widgets.IntSlider(
    value = 151,
    min = 0.1,
    max = 600,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

pressure_iteration_slider = widgets.IntRangeSlider(
    value = [0.1e5, 1.0e5],        # initial range [min_selected, max_selected]
    min = 0e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_objects():
  display(pressure_button)
  display(pressure_temp_slider)
  display(pressure_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_button.on_click(run_MC_pressure)

rebuild_pressure_objects()

Button(description='Run MC Simulation', layout=Layout(width='400px'), style=ButtonStyle(), tooltip='Run MC Sim…

IntSlider(value=151, continuous_update=False, description='Temperature:', layout=Layout(width='400px'), max=60…

IntRangeSlider(value=(10000, 100000), description='Iterations:', layout=Layout(width='400px'), max=10000000, r…

In [24]:
# @title Module: Pressure vs. Temperature

pressure_fraction_list = [('Ar', 1.0)]

'''
Calculates the pressure according to ideal gas law
density: density, in mols / L
temperature: temperature, in K

Returns: the pressure in pascals
'''
def ideal_gas_law(density, temperature):
  return density * R * temperature * 1000

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_pressure_temp(null_filler = None):
  #ref_model = LJRef(["argon"]) # note: this has to be hard-coded
  ideal_gas_law_array = []
  LJ_Ref_array = []

  clear_output()
  print("Running MC Simulation - Pressure Module")

  iterations = 10
  samples = 1

  RDF_pressure_array = np.zeros(iterations)
  virial_pressure_array = np.zeros(iterations)
  mayer_pressure_array = np.zeros(iterations)
  reference_pressure_array = []

  no_atoms = 256
  temperatures = np.linspace(130, 900, iterations)

  # consider making this a slider
  niterations = pressure_temp_iteration_slider.value[1]
  nwarmup = pressure_temp_iteration_slider.value[0]

  energy_arrays = []
  temp_sample_array_RDF = []
  temp_sample_array_virial = []
  temp_sample_array_mayer = []

  density_raw = 44.44 # mol / L
  density = density_raw * 1000 * 1e-10 * 1e-10 * 1e-10 * 6.022e23

  for temperature in temperatures:
    ideal_gas_law_array.append(ideal_gas_law(density_raw, temperature))

    #molar_vol_raw = 1 / (density_raw * 1000)
    #reference_pressure_array.append(reference_pressure(ref_model, molar_vol_raw, temperature))

  for i in range(len(temperatures)):
    for j in range(samples):
      temp = temperatures[i]
      Result = MC_main(density, no_atoms, pressure_fraction_list, niterations, temp, n_warmup = nwarmup, reference_start = True,
                       params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
      MC, RDF, R_bins, no_bins, energy_array, pressure, move_accepted_array = Result

      RDF_pressure_sample, virial_pressure_sample, mayer_pressure_sample = pressure
      temp_sample_array_RDF.append(RDF_pressure_sample)
      temp_sample_array_virial.append(virial_pressure_sample)
      temp_sample_array_mayer.append(mayer_pressure_sample)

    RDF_pressure = np.average(temp_sample_array_RDF)
    virial_pressure = np.average(temp_sample_array_virial)
    mayer_pressure = np.average(temp_sample_array_mayer)
    temp_sample_array_RDF.clear()
    temp_sample_array_virial.clear()
    temp_sample_array_mayer.clear()

    RDF_pressure_array[i] = RDF_pressure
    virial_pressure_array[i] = virial_pressure
    mayer_pressure_array[i] = mayer_pressure
    energy_arrays.append(energy_array)

  result_pressure_RDF_normal_units = []
  result_pressure_virial_normal_units = []
  result_pressure_mayer_normal_units = []

  for i in range(len(RDF_pressure_array)):
    result_pressure_RDF_normal_units.append(RDF_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_virial_normal_units.append(virial_pressure_array[i] * 1.602e-19 * 1e30)
    result_pressure_mayer_normal_units.append(mayer_pressure_array[i] * 1.602e-19 * 1e30)

  plt.scatter(temperatures, result_pressure_RDF_normal_units, label = 'RDF')
  plt.scatter(temperatures, result_pressure_virial_normal_units, label = 'Virial')
  plt.scatter(temperatures, result_pressure_mayer_normal_units, label = 'Mayer')
  plt.plot(temperatures, ideal_gas_law_array, label = 'Ideal Gas Law')
  #plt.plot(temperatures, reference_pressure_array, label = 'Reference LJ')
  plt.xlabel('Temperature (K)')
  plt.ylabel('Pressure (Pa)')
  plt.title('Pressure vs Temperature - Data')
  plt.legend()
  plt.show()

  iterations_array = np.zeros(niterations - nwarmup - 1)
  for i in range(len(iterations_array)):
    iterations_array[i] = i

  for i in range(len(energy_arrays)):
    energy_history = energy_arrays[i]
    plt.plot(iterations_array, energy_history, label = f'{i}')
  plt.xlabel('Iterations')
  plt.ylabel('Potential Energy (eV)')
  plt.title('Potential Energy vs Iteration')
  plt.legend()
  plt.show()

  rebuild_pressure_temp_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

pressure_temp_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

'''
Rebuilds all UI objects
'''
def rebuild_pressure_temp_objects():
  display(pressure_temp_button)
  display(pressure_temp_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
pressure_temp_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
pressure_temp_button.on_click(run_MC_pressure_temp)

rebuild_pressure_temp_objects()

Running MC Simulation - Pressure Module


TypeError: MC_main() got an unexpected keyword argument 'params'

In [25]:
# @title Module: Signal to Noise Ratio as Number of Atoms Increases

noise_fraction_list = [('Ar', 1.0)]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_noise(null_filler = None):
  clear_output()
  print("Running MC Simulation - Signal to Noise Module")

  # how many different N values will be tested
  iterations = 10
  samples = 1
  # how many MC simulations will be run for each N value to determine the SNR ratio

  final_average_RDF_array = []

  N_low = noise_N_slider.value[0]
  N_high = noise_N_slider.value[1]
  Ns = np.linspace(N_low, N_high, iterations)

  rho = 0.015
  temperature = 100
  niterations = noise_iteration_slider.value[1]
  nwarmup = noise_iteration_slider.value[0]

  signal_to_noise_array = np.zeros(iterations)
  temp_array = np.zeros(samples)

  for i in range(len(Ns)):
    SNR_accumulator = 0

    for j in range(samples):
      N = int(Ns[i])
      Result = MC_main(rho, N, noise_fraction_list, niterations, temperature, n_warmup = nwarmup,
                       params = LJ_avg_array, energy = lennard_jones, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction)
      MC, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      ASE_RDF, manual_RDF, _, _, _ = RDF
      final_average_RDF_array.append(manual_RDF)

    for k in range(no_bins):
      for h in range(samples):
        temp_array[h] = final_average_RDF_array[h][k] # takes the kth bin of every RDF for this iteration

      if(0 in temp_array):
        continue

      signal = np.average(temp_array)
      noise = np.std(temp_array)
      if(noise <= 0): noise = 1e-4

      SNR_accumulator += signal / noise
      # raw ratio because log was giving negative numbers

    signal_to_noise_array[i] = SNR_accumulator / samples
    final_average_RDF_array.clear()

  one_over_sqrt_N = 1 / np.sqrt(Ns)

  plt.plot(one_over_sqrt_N, signal_to_noise_array)
  plt.xlabel('1 / sqrt(N)')
  plt.ylabel('Signal to Noise Ratio')
  plt.title('Signal to Noise Ratio vs 1 / sqrt(N)')
  plt.show()

  rebuild_noise_objects()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)


noise_N_slider = widgets.IntRangeSlider(
    value = [256, 1024],        # initial range [min_selected, max_selected]
    min = 128,               # minimum possible value
    max = 2048,              # maximum possible value
    step = 32,
    description = 'Atoms:',
    style = style,
    layout = Layout(width = width)
)

noise_iteration_slider = widgets.IntRangeSlider(
    value = [1e5, 1e5],        # initial range [min_selected, max_selected]
    min = 1e5,               # minimum possible value
    max = 1e7,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_noise_objects():
  display(noise_button)
  display(noise_N_slider)
  display(noise_iteration_slider)

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
noise_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
noise_button.on_click(run_MC_noise)

rebuild_noise_objects()

Button(description='Run MC Simulation', layout=Layout(width='400px'), style=ButtonStyle(), tooltip='Run MC Sim…

IntRangeSlider(value=(256, 1024), description='Atoms:', layout=Layout(width='400px'), max=2048, min=128, step=…

IntRangeSlider(value=(100000, 100000), description='Iterations:', layout=Layout(width='400px'), max=10000000, …

In [91]:
# @title Module: Visualizing a Simulation

visual_fraction_list = [('Ar', 0.50, 'Atom'), ('Cl', 0.50, 'Atom')]

ASE_object_array = []
simulation_data_tuple_array = []
pre_rendered_images = []
visual_has_ran = False

# AI
def visual_render_frame(atoms, rotation = '0x, 0y, 0z'):
    buf = io.BytesIO()
    atoms_copy = atoms.copy()
    atoms_copy.set_cell(None)
    write(buf, atoms_copy, format = 'png', rotation = rotation)
    buf.seek(0)

    img = Image.open(buf)
    background = Image.new('RGBA', img.size, (255, 255, 255, 255))
    composite = Image.alpha_composite(background, img)

    out_buf = io.BytesIO()
    composite.save(out_buf, format='png')
    out_buf.seek(0)

    return out_buf.getvalue()

# AI
def visual_pre_render_all():
    global pre_rendered_images
    pre_rendered_images.clear()

    for i in tqdm(range(len(ASE_object_array)), desc = "Rendering Images:"):
        atoms = ASE_object_array[i].copy()
        img = visual_render_frame(atoms)
        pre_rendered_images.append(img)

    visual_slider.max = len(ASE_object_array) - 1
    rebuild_visual_objects()


def get_ASE_object(index):
  if(index < len(ASE_object_array)):
    return ASE_object_array[index]
  elif (index < 0):
    return ASE_object_array[0]
  else:
    return ASE_object_array[len(ASE_object_array) - 1]

'''
A wrapper function for the button. Runs the MC simulation according to what is inside of the function and loops
'''
def run_MC_visual(null_filler = None):
  clear_output()
  print("Running MC Simulation - Visualization")

  global ASE_object_array
  ASE_object_array.clear()
  rho = .5 * 6.022e23 * 1000 * 1e-10 * 1e-10 * 1e-10 # atoms / A^3
  no_atoms = 212

  temperature = visual_temp_slider.value
  niterations = visual_iteration_slider.value[1]
  nwarmup = visual_iteration_slider.value[0]
  iteration_step = 3000

  outer_iterations = niterations / iteration_step
  outer_iterations = int(outer_iterations)

  for i in range(outer_iterations):
    if(i == 0):
      Result = MC_main(rho, no_atoms, visual_fraction_list, 0, temperature, energy = lennard_jones, energy_derivative = lennard_jones_derivative, energy_params = LJ_avg_array,
                       n_warmup = nwarmup, cube_start = True, skip_final = True, move_step_size = 5.80)
      MC_return_tuple, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      MC, _, _, _, _ = MC_return_tuple

      ASE_object_array.append(MC.copy())
      simulation_data_tuple_array.append(deepcopy(MC_return_tuple))

    else:
      Result = MC_main(rho, no_atoms, visual_fraction_list, iteration_step, temperature,
                       energy_params = LJ_avg_array, n_warmup = nwarmup, cube_start = True, MC_initialized = deepcopy(simulation_data_tuple_array[-1]), skip_final = True)
      MC_return_tuple, RDF, R_bins, no_bins, heat_capacity, pressure, move_accepted_array = Result
      MC, _, _, _, _ = MC_return_tuple

      ASE_object_array.append(MC.copy())
      simulation_data_tuple_array.append(deepcopy(MC_return_tuple))

  global visual_has_ran
  visual_has_ran = True
  visual_pre_render_all()

style = {'description_width': '100px'}
width = '400px'
button_layout = Layout(width = width)

visual_temp_slider = widgets.IntSlider(
    value = 300,
    min = 200,
    max = 400,
    step = 10,
    description = 'Temperature:',
    disabled = False,
    continuous_update = False,
    orientation = 'horizontal',
    readout = True,
    readout_format = 'd',
    style = style,
    layout = Layout(width = width)
)

visual_iteration_slider = widgets.IntRangeSlider(
    value = [0e5, 6e5],
    min = 0e5,               # minimum possible value
    max = 10e5,              # maximum possible value
    step = 1e5,
    description = 'Iterations:',
    readout_format = '.1e',
    style = style,
    layout = Layout(width = width)
)

def rebuild_visual_objects():
  display(visual_button)
  display(visual_temp_slider)

  if(visual_has_ran):
    display(visual_image_box)
    if pre_rendered_images:
            visual_image_widget.value = pre_rendered_images[0]
            visual_slider.value = 0

'''
Creates a button and assigns it to run the MC simulation when clicked
'''
visual_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', icon = 'check', layout = button_layout)
visual_button.on_click(run_MC_visual)

visual_slider = IntSlider(
    value = 0,
    min = 0,
    max = 1,
    step = 1,
    description = 'Move Me',
    continuous_update = True,
    style = style,
    layout = Layout(width = width)
)

visual_image_widget = widgets.Image(format = 'png', width = 450, height = 450)

def visual_update_image(change):
    idx = change['new']
    if 0 <= idx < len(pre_rendered_images):
        visual_image_widget.value = pre_rendered_images[idx]

visual_slider.observe(visual_update_image, names = 'value')
visual_image_widget.layout.height = '450px'
visual_image_box = widgets.VBox([visual_slider, visual_image_widget])

clear_output()
rebuild_visual_objects()

Running MC Simulation - Visualization


Rendering Images:: 100%|██████████| 200/200 [00:21<00:00,  9.40it/s]


Button(description='Run MC Simulation', icon='check', layout=Layout(width='400px'), style=ButtonStyle(), toolt…

IntSlider(value=300, continuous_update=False, description='Temperature:', layout=Layout(width='400px'), max=40…

In [85]:
# @title Convergence Optimization

#equilibrium_fraction_list = [('Ar', 1.0)]
equilibrium_fraction_list = [('Ar', 0.50, 'Atom'), ('Cl', 0.50, 'Atom')]


def run_MC_equil(null_filler = None):
  clear_output()

  step_sizes = np.linspace(40.35, 75.55, 7)
  final_variance_array1 = []
  final_mean_array1 = []

  final_variance_array2 = []
  final_mean_array2 = []

  move_accepted_arrays = []
  moving_window_arrays = []

  N = 212
  # density = 1202.05 kg / m^3 (of Argon gas)
  raw_density = 1202.05 / 3 # kg / m^3 (Argon)
  density = 1.1 * raw_density * 1000 / 39.948 * 1e-10 * 1e-10 * 1e-10 * 6.022e23 # kg to grams, mols of Argon per gram, meters to A, mols to atoms
  density = .5 * 6.022e23 * 1000 * 1e-10 * 1e-10 * 1e-10 # atoms / A^3
  temperature = 300 # K

  iterations = int(0.2e6)
  warmup = int(0.1e5)

  for step_size in step_sizes:
    Result1 = MC_main(density, N, equilibrium_fraction_list, iterations, temperature, n_warmup = warmup, cube_start = True, move_step_size = step_size)
    MC_tuple1, RDF1, R_bins1, no_bins1, energy_array1, pressure1, move_accepted_array1 = Result1
    MC1, _, _, _, _ = MC_tuple1

    Result2 = MC_main(density, N, equilibrium_fraction_list, iterations, temperature, n_warmup = warmup, cube_start = True, move_step_size = step_size, anneal = True)
    MC_tuple2, RDF2, R_bins2, no_bins2, energy_array2, pressure2, move_accepted_array2 = Result2
    MC2, _, _, _, _ = MC_tuple2

    positions1 = MC1.get_positions()
    positions1 = positions1.tolist()

    positions2 = MC2.get_positions()
    positions2 = positions2.tolist()

    final_energy_values1 = energy_array1[-50000:]
    final_energy_values2 = energy_array2[-50000:]

    final_variance_array1.append(np.var(final_energy_values1))
    final_mean_array1.append(np.mean(final_energy_values1))

    final_variance_array2.append(np.var(final_energy_values2))
    final_mean_array2.append(np.mean(final_energy_values2))

    iteration_array = np.zeros(iterations - warmup)
    for a in range(len(iteration_array)):
      iteration_array[a] = a + warmup

    move_accepted_arrays.append(move_accepted_array1)

    plt.plot(iteration_array, energy_array1, label = f'default step = {step_size}')
    plt.xlabel('Iterations')
    plt.ylabel('Potential Energy (eV)')
    plt.title('Potential Energy vs Iteration')
    plt.legend()

  plt.autoscale(enable=True, axis='y', tight=True)
  plt.show()

  s = ', '.join(str(list(position)) for position in positions1)

  with open('equilibrium_positions.txt', 'w') as f:
    f.write('[' + s + ']')

  window_average_array1 = []
  window_average_array2 = []

  iteration_window_array = []

  for move_accept_array in move_accepted_arrays:
    for i in (range(1, iterations, 1000)):
      start = i - 1000
      if(start < 0): start = 0
      end = i

      moving_window_average1 = move_accepted_array1[start:end]
      moving_window_average2 = move_accepted_array2[start:end]

      accept_rate1 = np.average(moving_window_average1)
      accept_rate2 = np.average(moving_window_average2)

      window_average_array1.append(accept_rate1)
      window_average_array2.append(accept_rate2)

      iteration_window_array.append(i)

    plt.plot(iteration_window_array, window_average_array1, label = 'default')
    plt.legend()
    plt.xlabel('Iterations')
    plt.ylabel('Rate of Accepted Moves')
    plt.title('Rate of Accepted Moves vs Iteration')
    window_average_array1.clear()
    window_average_array2.clear()
    iteration_window_array.clear()

  plt.show()

  plt.plot(step_sizes, final_variance_array1, label = 'default')
  plt.legend()
  plt.xlabel('Step Size')
  plt.ylabel('Variance')
  plt.title('Variance vs Step Size')
  plt.show()

  plt.plot(step_sizes, final_mean_array1, label = 'default')
  plt.legend()
  plt.xlabel('Step Size')
  plt.ylabel('Mean')
  plt.title('Mean vs Step Size')
  plt.show()


  rebuild_equilibrium_objects()

button_layout = Layout(width = width)

equilibrium_button = widgets.Button(description = 'Run MC Simulation', disabled = False, button_style = '', tooltip = 'Run MC Simulation', layout = button_layout)
equilibrium_button.on_click(run_MC_equil)

def rebuild_equilibrium_objects():
  display(equilibrium_button)

rebuild_equilibrium_objects()


Button(description='Run MC Simulation', layout=Layout(width='400px'), style=ButtonStyle(), tooltip='Run MC Sim…

In [28]:
# @title OVITO test
# apparently OVITO live visualization crashed colab? (may be too much memory)

import ovito
from ovito.io import import_file
from ovito.vis import Viewport
from ovito.gui import create_qwidget, create_ipywidget
from PySide6.QtWidgets import QApplication
import sys


test = ovito.gui.create_ipywidget()

from ovito.gui import UtilityInterface
from ovito.traits import action_handler
from traits.api import Button

class MyUtilityApplet(UtilityInterface):
    hello_btn = Button(ovito_label="Click me")

    @action_handler("hello_btn")
    def say_hello(self):
        print("Hello world!")




'''
from ovito.io import import_file
from ovito.vis import Viewport
from ovito.gui import create_qwidget
from PySide6.QtWidgets import QApplication
import sys

# Import a particle model and add it to the global visualization scene.
#pipeline = import_file('output.extxyz')
#pipeline.add_to_scene()

# Create a virtual viewport.
vp = Viewport(type=Viewport.Type.Perspective, camera_dir=(2, 1, -1))

# Create a GUI widget associated with the viewport.
widget = create_qwidget(vp)
widget.resize(500, 400)
widget.setWindowTitle('OVITO Viewport Demo')
widget.show()
widget.raise_()
vp.zoom_all((widget.width(), widget.height()))

# Start the Qt event loop by invoking the QApplication.exec() method.
sys.exit(QApplication.instance().exec())
'''

"\nfrom ovito.io import import_file\nfrom ovito.vis import Viewport\nfrom ovito.gui import create_qwidget\nfrom PySide6.QtWidgets import QApplication\nimport sys\n\n# Import a particle model and add it to the global visualization scene.\n#pipeline = import_file('output.extxyz')\n#pipeline.add_to_scene()\n\n# Create a virtual viewport.\nvp = Viewport(type=Viewport.Type.Perspective, camera_dir=(2, 1, -1))\n\n# Create a GUI widget associated with the viewport.\nwidget = create_qwidget(vp)\nwidget.resize(500, 400)\nwidget.setWindowTitle('OVITO Viewport Demo')\nwidget.show()\nwidget.raise_()\nvp.zoom_all((widget.width(), widget.height()))\n\n# Start the Qt event loop by invoking the QApplication.exec() method.\nsys.exit(QApplication.instance().exec())\n"

In [29]:
# @title
from ovito.gui import UtilityInterface
from ovito.traits import action_handler
from traits.api import Button, Range
from ovito.traits import Vector3

class UtilityAppletWithParameters(UtilityInterface):

    frac = Range(low=0.0, high=1.0, value=0.3, label="Fract", ovito_group="Calculation")
    quality = Range(low=0.0, high=1.0, value=0.5, label="Quality", ovito_unit="percent", ovito_group="Calculation")
    radius = Range(low=0.0, value=0.0, label="Radius", ovito_placeholder="‹default›", ovito_group="Calculation")

    vec = Vector3((1.0, 0.5, 0.0), label="Direction", ovito_group="Displacement")

    run_button = Button(ovito_label="Run", ovito_group="Operation")

    @action_handler("run_button")
    def perform_operation(self):
        print(f"frac: {self.frac}")
        print(f"quality: {self.quality}")
        print(f"radius: {self.radius}")
        print(f"vec: {self.vec}")

In [30]:
# @title
# spare code, in case implmentation requires it again

'''
# separate dictionary for average LJ values
LJ_avg_dict = Dict.empty(key_type = types.unicode_type, value_type = types.UniTuple(types.float64, 2))

for key1 in LJ_dict.keys():
  for key2 in LJ_dict.keys():
    sigma1, epsilon1 = LJ_dict[key1]
    sigma2, epsilon2 = LJ_dict[key2]

    LJ_avg_dict[key1 + key2] = (((sigma1 + sigma2) / 2) ** 6, 4 * np.sqrt(epsilon1 * epsilon2))
'''

'\n# separate dictionary for average LJ values\nLJ_avg_dict = Dict.empty(key_type = types.unicode_type, value_type = types.UniTuple(types.float64, 2))\n\nfor key1 in LJ_dict.keys():\n  for key2 in LJ_dict.keys():\n    sigma1, epsilon1 = LJ_dict[key1]\n    sigma2, epsilon2 = LJ_dict[key2]\n\n    LJ_avg_dict[key1 + key2] = (((sigma1 + sigma2) / 2) ** 6, 4 * np.sqrt(epsilon1 * epsilon2))\n'

In [31]:
# @title
#testing cell
'''
import ase
from ase import Atoms, Atom
from ase.calculators.lj import LennardJones as LJ
from ase.geometry.rdf import get_rdf
from ase.build import molecule

import matplotlib.pyplot as plt
from ase.visualize.plot import plot_atoms

import numpy as np
import random

test1 = Atoms("H2", [[0, 0, 0], [1.0, 0, 0]])
test2 = Atoms("C", [[0, 1.5, 0]])
test3 = Atoms("B2", [[0, 1.5, 0], [9, 9, 9]])
test4 = Atoms([Atom('C', [0., 2., 0.]),
               Atom('O', [1.1, 0., 3.])],
              cell=(10, 10, 10))
test5 = molecule("SiH4")
test6 = molecule("H2O2")
test7 = molecule("H2O")

test = test1 + test2 + test3 + test4 + test5 + test6 + test7
print(test)
print(test.get_positions())
print(test.get_atomic_numbers())

fig, ax = plt.subplots()
plot_atoms(test, ax, rotation='45x,45y,45z')
plt.show()
'''

'\nimport ase\nfrom ase import Atoms, Atom\nfrom ase.calculators.lj import LennardJones as LJ\nfrom ase.geometry.rdf import get_rdf\nfrom ase.build import molecule\n\nimport matplotlib.pyplot as plt\nfrom ase.visualize.plot import plot_atoms\n\nimport numpy as np\nimport random\n\ntest1 = Atoms("H2", [[0, 0, 0], [1.0, 0, 0]])\ntest2 = Atoms("C", [[0, 1.5, 0]])\ntest3 = Atoms("B2", [[0, 1.5, 0], [9, 9, 9]])\ntest4 = Atoms([Atom(\'C\', [0., 2., 0.]),\n               Atom(\'O\', [1.1, 0., 3.])],\n              cell=(10, 10, 10))\ntest5 = molecule("SiH4")\ntest6 = molecule("H2O2")\ntest7 = molecule("H2O")\n\ntest = test1 + test2 + test3 + test4 + test5 + test6 + test7\nprint(test)\nprint(test.get_positions())\nprint(test.get_atomic_numbers())\n\nfig, ax = plt.subplots()\nplot_atoms(test, ax, rotation=\'45x,45y,45z\')\nplt.show()\n'